### 📈 Arithmetic and Geometric Brownian Motion Masterclass

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Learn to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### 📈 Visualizing Arithmetic and Geometric Brownian Motion
 
   $$
   \begin{array}{rclcl}
     \text{ABM:}      &\quad dS_t = \mu\,dt + \sigma\,dW_t        &\qquad\Big\backslash\qquad&    \text{GBM:}      &\quad \dfrac{dS_t}{S_t} = \mu\,dt + \sigma\,dW_t
   \end{array}
   $$
 Literally the only difference between the two is that geometric is multiplicative and arithmetic is additive.

 Additive returns won't exhibit volatility drag like with multiplicative returns, just strict arithmetic (linear) growth.

###### ______________________________________________________________________________________________________________________________________

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

YEARS = 15
STEPS_PER_YEAR = 12              # monthly steps
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"
N_PATHS = 30

MU = 0.30                        # 10% annual drift
SIGMA = 0.22                     # 22% annual volatility

FRAME_STRIDE = 3                 # reveal quarterly
FRAME_DURATION = 45
INITIAL_I = 3

OUTPUT_HTML = "abm_vs_gbm_arithmetic_vs_geometric_mean.html"
SHOW_FIG = True

# ============================================================
# Simulation helpers
# ============================================================

def simulate_abm_path(x0, mu, sigma, n_steps, rng):
    """
    Arithmetic Brownian Motion:
        dX_t = mu dt + sigma dW_t
    """
    z = rng.normal(size=n_steps)
    increments = mu * DT + sigma * np.sqrt(DT) * z
    return x0 + np.r_[0.0, np.cumsum(increments)]


def simulate_gbm_path(s0, mu, sigma, n_steps, rng):
    """
    Geometric Brownian Motion:
        dS_t = mu S_t dt + sigma S_t dW_t
    """
    z = rng.normal(size=n_steps)
    log_returns = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * z
    )
    return s0 * np.exp(np.r_[0.0, np.cumsum(log_returns)])


def abm_theoretical_mean(x0, mu, times):
    return x0 + mu * times


def gbm_theoretical_arithmetic_mean(s0, mu, times):
    return s0 * np.exp(mu * times)


def gbm_theoretical_geometric_mean(s0, mu, sigma, times):
    """
    exp(E[log S_t]) for GBM, which is also the median path.
    """
    return s0 * np.exp((mu - 0.5 * sigma**2) * times)


def padded_range(values, pad_fraction=0.08, min_pad=0.10):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [v_min - pad, v_max + pad]


# ============================================================
# Time axis
# ============================================================

dates = pd.date_range(start=START_DATE, periods=N_STEPS + 1, freq="MS")
times = np.arange(N_STEPS + 1) * DT

# ============================================================
# Simulate ABM and GBM paths
# ============================================================

abm_paths = np.column_stack([
    simulate_abm_path(INITIAL_VALUE, MU, SIGMA, N_STEPS, rng)
    for _ in range(N_PATHS)
])

gbm_paths = np.column_stack([
    simulate_gbm_path(INITIAL_VALUE, MU, SIGMA, N_STEPS, rng)
    for _ in range(N_PATHS)
])

abm_mean = abm_theoretical_mean(INITIAL_VALUE, MU, times)
gbm_arith_mean = gbm_theoretical_arithmetic_mean(INITIAL_VALUE, MU, times)
gbm_geom_mean = gbm_theoretical_geometric_mean(INITIAL_VALUE, MU, SIGMA, times)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

abm_mean_color = "#ffaa33"       # orange
gbm_arith_color = "#00ff88"      # green
gbm_geom_color = "#ffd84d"       # yellow

path_color_over = "#18d618"      # green
path_color_under = "#ff3030"     # red

baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.08,
    subplot_titles=(
        "Arithmetic Brownian Motion",
        "Geometric Brownian Motion",
    ),
)

initial_end = min(INITIAL_I, N_STEPS)

# ------------------------------------------------------------
# Left panel: ABM paths
# ------------------------------------------------------------

for j in range(N_PATHS):
    yvals = abm_paths[: initial_end + 1, j]
    meanvals = abm_mean[: initial_end + 1]
    color = path_color_over if yvals[-1] > meanvals[-1] else path_color_under

    fig.add_trace(
        go.Scatter(
            x=dates[: initial_end + 1],
            y=yvals,
            mode="lines",
            line=dict(color=color, width=1.5),
            opacity=0.50,
            name="30 ABM simulations" if j == 0 else f"ABM path {j+1}",
            legendgroup="abm-paths",
            showlegend=(j == 0),
            hovertemplate=(
                f"ABM path {j+1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=dates[: initial_end + 1],
        y=abm_mean[: initial_end + 1],
        mode="lines",
        line=dict(color=abm_mean_color, width=4),
        name="Theoretical mean",
        legendgroup="abm-mean",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "ABM mean: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

# ------------------------------------------------------------
# Right panel: GBM paths
# ------------------------------------------------------------

for j in range(N_PATHS):
    yvals = gbm_paths[: initial_end + 1, j]
    meanvals = gbm_arith_mean[: initial_end + 1]
    color = path_color_over if yvals[-1] > meanvals[-1] else path_color_under

    fig.add_trace(
        go.Scatter(
            x=dates[: initial_end + 1],
            y=yvals,
            mode="lines",
            line=dict(color=color, width=1.5),
            opacity=0.50,
            name="30 GBM simulations" if j == 0 else f"GBM path {j+1}",
            legendgroup="gbm-paths",
            showlegend=(j == 0),
            hovertemplate=(
                f"GBM path {j+1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

fig.add_trace(
    go.Scatter(
        x=dates[: initial_end + 1],
        y=gbm_arith_mean[: initial_end + 1],
        mode="lines",
        line=dict(color=gbm_arith_color, width=4),
        name="Theoretical arithmetic mean",
        legendgroup="gbm-arith-mean",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "E[S_t]: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=dates[: initial_end + 1],
        y=gbm_geom_mean[: initial_end + 1],
        mode="lines",
        line=dict(color=gbm_geom_color, width=4, dash="dash"),
        name="Theoretical geometric mean",
        legendgroup="gbm-geom-mean",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "exp(E[log S_t]): %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# Baseline at initial value
fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1,
)

fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(INITIAL_I, N_STEPS + 1, FRAME_STRIDE))
if frame_indices[-1] != N_STEPS:
    frame_indices.append(N_STEPS)

n_left_traces = N_PATHS + 1
n_right_traces = N_PATHS + 2
n_total_traces = n_left_traces + n_right_traces

for i in frame_indices:
    frame_name = f"f{i}"
    frame_data = []

    # Left panel: ABM paths
    for j in range(N_PATHS):
        yvals = abm_paths[: i + 1, j]
        meanvals = abm_mean[: i + 1]
        color = path_color_over if yvals[-1] > meanvals[-1] else path_color_under

        frame_data.append(
            go.Scatter(
                x=dates[: i + 1],
                y=yvals,
                mode="lines",
                opacity=0.50,
                line=dict(color=color, width=1.5),
            )
        )

    # Left panel: ABM theoretical mean
    frame_data.append(
        go.Scatter(
            x=dates[: i + 1],
            y=abm_mean[: i + 1],
            mode="lines",
            line=dict(color=abm_mean_color, width=4),
        )
    )

    # Right panel: GBM paths
    for j in range(N_PATHS):
        yvals = gbm_paths[: i + 1, j]
        meanvals = gbm_arith_mean[: i + 1]
        color = path_color_over if yvals[-1] > meanvals[-1] else path_color_under

        frame_data.append(
            go.Scatter(
                x=dates[: i + 1],
                y=yvals,
                mode="lines",
                opacity=0.50,
                line=dict(color=color, width=1.5),
            )
        )

    # Right panel: GBM theoretical arithmetic mean
    frame_data.append(
        go.Scatter(
            x=dates[: i + 1],
            y=gbm_arith_mean[: i + 1],
            mode="lines",
            line=dict(color=gbm_arith_color, width=4),
        )
    )

    # Right panel: GBM theoretical geometric mean
    frame_data.append(
        go.Scatter(
            x=dates[: i + 1],
            y=gbm_geom_mean[: i + 1],
            mode="lines",
            line=dict(color=gbm_geom_color, width=4, dash="dash"),
        )
    )

    frames.append(
        go.Frame(
            data=frame_data,
            traces=list(range(n_total_traces)),
            name=frame_name,
        )
    )

    elapsed_years = i / STEPS_PER_YEAR
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Axis ranges
# ============================================================

left_values = np.r_[abm_paths.ravel(), abm_mean]
right_values = np.r_[gbm_paths.ravel(), gbm_arith_mean, gbm_geom_mean]

left_y_range = padded_range(left_values, pad_fraction=0.08, min_pad=0.20)
right_y_range = padded_range(right_values, pad_fraction=0.08, min_pad=0.20)

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="Arithmetic vs Geometric Brownian Motion",
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700,
    width=1280,
    margin=dict(t=100, b=120, r=40, l=70),
    legend=dict(
        orientation="v",
        x=0.0,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 70},
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": 0.0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "visible": False   # removes the small subtext above the slider
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 40},
        "len": 0.85,
        "x": 0.15,
        "y": 0.0,
        "steps": slider_steps,
    }],
)

# Style subplot titles
fig.update_annotations(font=dict(color=off_white, size=16))

# Axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[dates[0], dates[-1]],
    title_text="Date",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=left_y_range,
    title_text="Value",
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[dates[0], dates[-1]],
    title_text="Date",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=right_y_range,
    title_text="Value",
)

# ============================================================
# Save / show
# ============================================================
fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Understand that These Are just Random Variables


   $$
   \begin{array}{rclcl}
     \text{ABM:}      &\quad S_t = S_0 + \mu t + \sigma W_t &\qquad\Big\backslash\qquad&
     \text{GBM:}      &\quad S_t = S_0 \exp\left((\mu - \frac{1}{2}\sigma^2)t + \sigma W_t\right) \\
     \text{ABM Distribution:} &\quad S_t \sim \mathcal{N}(S_0 + \mu t,\, \sigma^2 t)\qquad &\Big\backslash\qquad&
     \text{GBM Distribution:} &\quad \ln S_t \sim \mathcal{N}\left(\ln S_0 + (\mu - \frac{1}{2}\sigma^2)t,\, \sigma^2 t\right)
   \end{array}
   $$

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

YEARS = 15
STEPS_PER_YEAR = 12                 # monthly simulation steps
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1.0 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

N_DISPLAY_PATHS = 30                # paths drawn in the upper panels
N_DISTRIBUTION_PATHS = 5_000        # paths used to form each histogram

MU = 0.30                           # 30% annual drift
SIGMA = 0.22                        # 22% annual volatility

FRAME_STRIDE = 3                    # one animation frame per quarter
FRAME_DURATION = 45
INITIAL_I = 3                       # first frame at 0.25 years

N_HIST_BINS = 48
N_PDF_POINTS = 500
PDF_Z_RANGE = 3.75
DENSITY_HEADROOM = 1.15             # 15% headroom above each frame's peak

SHOW_FIG = True

# ============================================================
# Simulation helpers
# ============================================================


def simulate_abm_paths(x0, mu, sigma, n_steps, n_paths, rng):
    shocks = rng.normal(size=(n_steps, n_paths))
    increments = mu * DT + sigma * np.sqrt(DT) * shocks
    cumulative_increments = np.vstack([
        np.zeros((1, n_paths)),
        np.cumsum(increments, axis=0),
    ])
    return x0 + cumulative_increments


def simulate_gbm_paths(s0, mu, sigma, n_steps, n_paths, rng):
    shocks = rng.normal(size=(n_steps, n_paths))
    log_increments = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * shocks
    )
    cumulative_log_returns = np.vstack([
        np.zeros((1, n_paths)),
        np.cumsum(log_increments, axis=0),
    ])
    return s0 * np.exp(cumulative_log_returns)


def abm_theoretical_mean(x0, mu, times):
    return x0 + mu * times


def gbm_theoretical_arithmetic_mean(s0, mu, times):
    return s0 * np.exp(mu * times)


def gbm_theoretical_geometric_mean(s0, mu, sigma, times):
    return s0 * np.exp((mu - 0.5 * sigma**2) * times)


def normal_pdf(x, mean, sd):
    x = np.asarray(x, dtype=float)
    return (
        np.exp(-0.5 * ((x - mean) / sd) ** 2)
        / (sd * np.sqrt(2.0 * np.pi))
    )


def lognormal_pdf(x, log_mean, log_sd):
    x = np.asarray(x, dtype=float)
    density = np.zeros_like(x)
    positive = x > 0.0
    xp = x[positive]
    density[positive] = (
        np.exp(-0.5 * ((np.log(xp) - log_mean) / log_sd) ** 2)
        / (xp * log_sd * np.sqrt(2.0 * np.pi))
    )
    return density


def histogram_density(samples, x_min, x_max, n_bins):
    edges = np.linspace(x_min, x_max, n_bins + 1)
    counts, _ = np.histogram(samples, bins=edges)
    widths = np.diff(edges)
    centers = edges[:-1] + 0.5 * widths
    density = counts / (len(samples) * widths)
    return centers, density, widths


def padded_range(values, pad_fraction=0.08, min_pad=0.10, floor=None):
    values = np.asarray(values, dtype=float)
    value_min = float(np.nanmin(values))
    value_max = float(np.nanmax(values))
    if np.isclose(value_min, value_max):
        pad = max(abs(value_max) * pad_fraction, min_pad)
    else:
        pad = max((value_max - value_min) * pad_fraction, min_pad)
    lower = value_min - pad
    upper = value_max + pad
    if floor is not None:
        lower = max(floor, lower)
    return [lower, upper]

# ============================================================
# Time axis and simulations
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)
times = np.arange(N_STEPS + 1) * DT

abm_paths = simulate_abm_paths(
    INITIAL_VALUE,
    MU,
    SIGMA,
    N_STEPS,
    N_DISTRIBUTION_PATHS,
    rng,
)

gbm_paths = simulate_gbm_paths(
    INITIAL_VALUE,
    MU,
    SIGMA,
    N_STEPS,
    N_DISTRIBUTION_PATHS,
    rng,
)

abm_display_paths = abm_paths[:, :N_DISPLAY_PATHS]
gbm_display_paths = gbm_paths[:, :N_DISPLAY_PATHS]

abm_mean = abm_theoretical_mean(INITIAL_VALUE, MU, times)
gbm_arithmetic_mean = gbm_theoretical_arithmetic_mean(
    INITIAL_VALUE,
    MU,
    times,
)
gbm_geometric_mean = gbm_theoretical_geometric_mean(
    INITIAL_VALUE,
    MU,
    SIGMA,
    times,
)

# ============================================================
# Horizon-distribution builders
# ============================================================

def abm_distribution_at(step):
    t = times[step]
    mean = MU * t
    sd = SIGMA * np.sqrt(t)
    x_min = mean - PDF_Z_RANGE * sd
    x_max = mean + PDF_Z_RANGE * sd
    samples = abm_paths[step, :] - INITIAL_VALUE
    hist_x, hist_y, hist_width = histogram_density(
        samples,
        x_min,
        x_max,
        N_HIST_BINS,
    )
    pdf_x = np.linspace(x_min, x_max, N_PDF_POINTS)
    pdf_y = normal_pdf(pdf_x, mean, sd)
    density_ceiling = DENSITY_HEADROOM * max(
        float(np.max(hist_y)),
        float(np.max(pdf_y)),
    )
    return {
        "hist_x": hist_x,
        "hist_y": hist_y,
        "hist_width": hist_width,
        "pdf_x": pdf_x,
        "pdf_y": pdf_y,
        "mean": mean,
        "x_range": [x_min, x_max],
        "y_range": [0.0, density_ceiling],
        "density_ceiling": density_ceiling,
    }


def gbm_distribution_at(step):
    t = times[step]
    log_mean = (MU - 0.5 * SIGMA**2) * t
    log_sd = SIGMA * np.sqrt(t)
    x_min = np.exp(log_mean - PDF_Z_RANGE * log_sd)
    x_max = np.exp(log_mean + PDF_Z_RANGE * log_sd)
    samples = gbm_paths[step, :] / INITIAL_VALUE
    hist_x, hist_y, hist_width = histogram_density(
        samples,
        x_min,
        x_max,
        N_HIST_BINS,
    )
    pdf_x = np.linspace(x_min, x_max, N_PDF_POINTS)
    pdf_y = lognormal_pdf(pdf_x, log_mean, log_sd)
    density_ceiling = DENSITY_HEADROOM * max(
        float(np.max(hist_y)),
        float(np.max(pdf_y)),
    )
    return {
        "hist_x": hist_x,
        "hist_y": hist_y,
        "hist_width": hist_width,
        "pdf_x": pdf_x,
        "pdf_y": pdf_y,
        "arithmetic_mean": np.exp(MU * t),
        "geometric_mean": np.exp(log_mean),
        "x_range": [x_min, x_max],
        "y_range": [0.0, density_ceiling],
        "density_ceiling": density_ceiling,
    }

# ============================================================
# Animation frame schedule
# ============================================================

frame_indices = list(range(INITIAL_I, N_STEPS + 1, FRAME_STRIDE))
if frame_indices[-1] != N_STEPS:
    frame_indices.append(N_STEPS)

initial_step = frame_indices[0]
initial_abm_distribution = abm_distribution_at(initial_step)
initial_gbm_distribution = gbm_distribution_at(initial_step)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
abm_mean_color = "#ffaa33"
gbm_arithmetic_color = "#00ff88"
gbm_geometric_color = "#ffd84d"
path_color_above = "#18d618"
path_color_below = "#ff3030"
histogram_fill = "rgba(0,212,255,0.45)"
histogram_edge = "rgba(0,212,255,0.75)"
pdf_color = "#f2f2f2"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# For vertical indicator line
spot_line_color = "#f4b200"
spot_line_style = "dash"
spot_line_width = 2.5
spot_line_opacity = 0.6

# ============================================================
# Figure structure
# ============================================================

fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.50, 0.50],
    row_heights=[0.62, 0.38],
    horizontal_spacing=0.08,
    vertical_spacing=0.13,
    subplot_titles=(
        "Arithmetic Brownian Motion",
        "Geometric Brownian Motion",
        "Normal cumulative-increment distribution",
        "Lognormal gross-return distribution",
    ),
)

# ============================================================
# Initial traces: upper-left ABM panel
# ============================================================

for path_index in range(N_DISPLAY_PATHS):
    path_values = abm_display_paths[: initial_step + 1, path_index]
    path_color = (
        path_color_above
        if path_values[-1] > abm_mean[initial_step]
        else path_color_below
    )

    fig.add_trace(
        go.Scatter(
            x=dates[: initial_step + 1],
            y=path_values,
            mode="lines",
            line=dict(color=path_color, width=1.5),
            opacity=0.50,
            name=(
                f"{N_DISPLAY_PATHS} ABM simulations"
                if path_index == 0
                else f"ABM path {path_index + 1}"
            ),
            legendgroup="abm-paths",
            showlegend=False,
            hovertemplate=(
                f"ABM path {path_index + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=dates[: initial_step + 1],
        y=abm_mean[: initial_step + 1],
        mode="lines",
        line=dict(color=abm_mean_color, width=4),
        name="ABM theoretical mean",
        legendgroup="abm-mean",
        showlegend=False,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "E[Xₜ]: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

# Fix: Use Scatter for vline instead of add_vline to avoid Timestamp addition errors
fig.add_trace(
    go.Scatter(
        x=[dates[initial_step], dates[initial_step]],
        y=[None, None],  # This will be set after abm_path_range is calculated.
        mode="lines",
        line=dict(color=spot_line_color, width=spot_line_width, dash=spot_line_style),
        opacity=spot_line_opacity,
        showlegend=False,
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)

# ============================================================
# Initial traces: upper-right GBM panel
# ============================================================

for path_index in range(N_DISPLAY_PATHS):
    path_values = gbm_display_paths[: initial_step + 1, path_index]
    path_color = (
        path_color_above
        if path_values[-1] > gbm_arithmetic_mean[initial_step]
        else path_color_below
    )

    fig.add_trace(
        go.Scatter(
            x=dates[: initial_step + 1],
            y=path_values,
            mode="lines",
            line=dict(color=path_color, width=1.5),
            opacity=0.50,
            name=(
                f"{N_DISPLAY_PATHS} GBM simulations"
                if path_index == 0
                else f"GBM path {path_index + 1}"
            ),
            legendgroup="gbm-paths",
            showlegend=False,
            hovertemplate=(
                f"GBM path {path_index + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

fig.add_trace(
    go.Scatter(
        x=dates[: initial_step + 1],
        y=gbm_arithmetic_mean[: initial_step + 1],
        mode="lines",
        line=dict(color=gbm_arithmetic_color, width=4),
        name="GBM theoretical arithmetic mean",
        legendgroup="gbm-arithmetic-mean",
        showlegend=False,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "E[Sₜ]: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=dates[: initial_step + 1],
        y=gbm_geometric_mean[: initial_step + 1],
        mode="lines",
        line=dict(color=gbm_geometric_color, width=4, dash="dash"),
        name="GBM theoretical geometric mean",
        legendgroup="gbm-geometric-mean",
        showlegend=False,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "exp(E[log Sₜ]): %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# Fix: Use Scatter for vline instead of add_vline to avoid Timestamp addition errors
fig.add_trace(
    go.Scatter(
        x=[dates[initial_step], dates[initial_step]],
        y=[None, None],  # This will be set after gbm_path_range is calculated.
        mode="lines",
        line=dict(color=spot_line_color, width=spot_line_width, dash=spot_line_style),
        opacity=spot_line_opacity,
        showlegend=False,
        hoverinfo="skip",
    ),
    row=1,
    col=2,
)

# ============================================================
# Initial traces: lower-left ABM distribution
# ============================================================

fig.add_trace(
    go.Bar(
        x=initial_abm_distribution["hist_x"],
        y=initial_abm_distribution["hist_y"],
        width=initial_abm_distribution["hist_width"],
        marker=dict(
            color=histogram_fill,
            line=dict(color=histogram_edge, width=0.5),
        ),
        name="ABM simulated horizon increments",
        legendgroup="abm-distribution",
        showlegend=False,
        hovertemplate=(
            "Increment: %{x:.4f}<br>"
            "Density: %{y:.4f}<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=initial_abm_distribution["pdf_x"],
        y=initial_abm_distribution["pdf_y"],
        mode="lines",
        line=dict(color=pdf_color, width=3),
        name="Theoretical normal PDF",
        legendgroup="abm-distribution",
        showlegend=False,
        hovertemplate=(
            "Increment: %{x:.4f}<br>"
            "Normal PDF: %{y:.4f}<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[
            initial_abm_distribution["mean"],
            initial_abm_distribution["mean"],
        ],
        y=[0.0, initial_abm_distribution["density_ceiling"]],
        mode="lines",
        line=dict(color=abm_mean_color, width=2, dash="dash"),
        showlegend=False,
        hovertemplate="ABM increment mean: %{x:.4f}<extra></extra>",
    ),
    row=2,
    col=1,
)

# ============================================================
# Initial traces: lower-right GBM distribution
# ============================================================

fig.add_trace(
    go.Bar(
        x=initial_gbm_distribution["hist_x"],
        y=initial_gbm_distribution["hist_y"],
        width=initial_gbm_distribution["hist_width"],
        marker=dict(
            color=histogram_fill,
            line=dict(color=histogram_edge, width=0.5),
        ),
        name="GBM simulated gross returns",
        legendgroup="gbm-distribution",
        showlegend=False,
        hovertemplate=(
            "Gross return: %{x:.4f}×<br>"
            "Density: %{y:.4f}<extra></extra>"
        ),
    ),
    row=2,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=initial_gbm_distribution["pdf_x"],
        y=initial_gbm_distribution["pdf_y"],
        mode="lines",
        line=dict(color=pdf_color, width=3),
        name="Theoretical lognormal PDF",
        legendgroup="gbm-distribution",
        showlegend=False,
        hovertemplate=(
            "Gross return: %{x:.4f}×<br>"
            "Lognormal PDF: %{y:.4f}<extra></extra>"
        ),
    ),
    row=2,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[
            initial_gbm_distribution["arithmetic_mean"],
            initial_gbm_distribution["arithmetic_mean"],
        ],
        y=[0.0, initial_gbm_distribution["density_ceiling"]],
        mode="lines",
        line=dict(color=gbm_arithmetic_color, width=2, dash="dash"),
        showlegend=False,
        hovertemplate="Arithmetic mean: %{x:.4f}×<extra></extra>",
    ),
    row=2,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[
            initial_gbm_distribution["geometric_mean"],
            initial_gbm_distribution["geometric_mean"],
        ],
        y=[0.0, initial_gbm_distribution["density_ceiling"]],
        mode="lines",
        line=dict(color=gbm_geometric_color, width=2, dash="dash"),
        showlegend=False,
        hovertemplate="Geometric mean: %{x:.4f}×<extra></extra>",
    ),
    row=2,
    col=2,
)

# Baselines in the path panels
fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1,
)

fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

n_upper_traces = (
    N_DISPLAY_PATHS + 1
    + N_DISPLAY_PATHS + 2
)
n_lower_traces = 3 + 4
n_animated_traces = n_upper_traces + n_lower_traces
animated_trace_indices = list(range(n_animated_traces + 2))  # Add +2 for new vlines

for step in frame_indices:
    frame_name = f"f{step}"
    frame_data = []

    # Upper-left ABM paths
    for path_index in range(N_DISPLAY_PATHS):
        path_values = abm_display_paths[: step + 1, path_index]
        path_color = (
            path_color_above
            if path_values[-1] > abm_mean[step]
            else path_color_below
        )

        frame_data.append(
            go.Scatter(
                x=dates[: step + 1],
                y=path_values,
                mode="lines",
                opacity=0.50,
                line=dict(color=path_color, width=1.5),
            )
        )

    frame_data.append(
        go.Scatter(
            x=dates[: step + 1],
            y=abm_mean[: step + 1],
            mode="lines",
            line=dict(color=abm_mean_color, width=4),
        )
    )

    # Vertical vline for spot (ABM panel)
    frame_data.append(
        go.Scatter(
            x=[dates[step], dates[step]],
            y=[None, None],  # to be set later after abm_path_range defined
            mode="lines",
            line=dict(color=spot_line_color, width=spot_line_width, dash=spot_line_style),
            opacity=spot_line_opacity,
            hoverinfo="skip",
            showlegend=False,
        )
    )

    # Upper-right GBM paths
    for path_index in range(N_DISPLAY_PATHS):
        path_values = gbm_display_paths[: step + 1, path_index]
        path_color = (
            path_color_above
            if path_values[-1] > gbm_arithmetic_mean[step]
            else path_color_below
        )

        frame_data.append(
            go.Scatter(
                x=dates[: step + 1],
                y=path_values,
                mode="lines",
                opacity=0.50,
                line=dict(color=path_color, width=1.5),
            )
        )

    frame_data.append(
        go.Scatter(
            x=dates[: step + 1],
            y=gbm_arithmetic_mean[: step + 1],
            mode="lines",
            line=dict(color=gbm_arithmetic_color, width=4),
        )
    )

    frame_data.append(
        go.Scatter(
            x=dates[: step + 1],
            y=gbm_geometric_mean[: step + 1],
            mode="lines",
            line=dict(color=gbm_geometric_color, width=4, dash="dash"),
        )
    )

    # Vertical vline for spot (GBM panel)
    frame_data.append(
        go.Scatter(
            x=[dates[step], dates[step]],
            y=[None, None],  # to be set later after gbm_path_range defined
            mode="lines",
            line=dict(color=spot_line_color, width=spot_line_width, dash=spot_line_style),
            opacity=spot_line_opacity,
            hoverinfo="skip",
            showlegend=False,
        )
    )

    abm_distribution = abm_distribution_at(step)
    gbm_distribution = gbm_distribution_at(step)

    frame_data.append(
        go.Bar(
            x=abm_distribution["hist_x"],
            y=abm_distribution["hist_y"],
            width=abm_distribution["hist_width"],
        )
    )

    frame_data.append(
        go.Scatter(
            x=abm_distribution["pdf_x"],
            y=abm_distribution["pdf_y"],
            mode="lines",
        )
    )

    frame_data.append(
        go.Scatter(
            x=[abm_distribution["mean"], abm_distribution["mean"]],
            y=[0.0, abm_distribution["density_ceiling"]],
            mode="lines",
        )
    )

    frame_data.append(
        go.Bar(
            x=gbm_distribution["hist_x"],
            y=gbm_distribution["hist_y"],
            width=gbm_distribution["hist_width"],
        )
    )

    frame_data.append(
        go.Scatter(
            x=gbm_distribution["pdf_x"],
            y=gbm_distribution["pdf_y"],
            mode="lines",
        )
    )

    frame_data.append(
        go.Scatter(
            x=[
                gbm_distribution["arithmetic_mean"],
                gbm_distribution["arithmetic_mean"],
            ],
            y=[0.0, gbm_distribution["density_ceiling"]],
            mode="lines",
        )
    )

    frame_data.append(
        go.Scatter(
            x=[
                gbm_distribution["geometric_mean"],
                gbm_distribution["geometric_mean"],
            ],
            y=[0.0, gbm_distribution["density_ceiling"]],
            mode="lines",
        )
    )

    frame_layout = go.Layout(
        xaxis3=dict(range=abm_distribution["x_range"]),
        yaxis3=dict(range=abm_distribution["y_range"]),
        xaxis4=dict(range=gbm_distribution["x_range"]),
        yaxis4=dict(range=gbm_distribution["y_range"]),
    )

    frames.append(
        go.Frame(
            data=frame_data,
            traces=animated_trace_indices,
            layout=frame_layout,
            name=frame_name,
        )
    )

    elapsed_years = step / STEPS_PER_YEAR
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": True},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Static ranges for the upper path panels
# ============================================================

abm_path_range = padded_range(
    np.r_[abm_display_paths.ravel(), abm_mean],
    pad_fraction=0.08,
    min_pad=0.20,
)

gbm_path_range = padded_range(
    np.r_[
        gbm_display_paths.ravel(),
        gbm_arithmetic_mean,
        gbm_geometric_mean,
    ],
    pad_fraction=0.08,
    min_pad=0.20,
    floor=0.0,
)

# Set y values for initial vlines now that ranges are available
# The vline traces are the last trace added to each subplot in the above code.
from plotly.basedatatypes import BaseFigure

def set_vline_y(trace, y_range):
    trace.y = [y_range[0], y_range[1]]

# Set for ABM (subplot 1,1)
for t in fig.data:
    # Use the knowledge that vline traces have both x values 
    # equal and y values [None, None] initially and `hoverinfo="skip"`.
    if isinstance(t, go.Scatter) and t.hoverinfo == "skip" and t.y == (None, None):
        if t.x[0] == t.x[1] == dates[initial_step]:
            if t in fig.select_traces(row=1, col=1):
                set_vline_y(t, abm_path_range)
            elif t in fig.select_traces(row=1, col=2):
                set_vline_y(t, gbm_path_range)

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="Brownian Motion Paths and Horizon Distributions",
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=940,
    width=1320,
    margin=dict(t=105, b=125, r=40, l=75),
    showlegend=False,
    hovermode="closest",
    bargap=0.03,
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": True,
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 70},
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": 0.0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {"visible": False},
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 40},
        "len": 0.85,
        "x": 0.15,
        "y": 0.0,
        "steps": slider_steps,
    }],
)

fig.update_annotations(font=dict(color=off_white, size=15))

# Upper axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[dates[0], dates[-1]],
    title_text="Date",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=abm_path_range,
    title_text="Value",
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[dates[0], dates[-1]],
    title_text="Date",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=gbm_path_range,
    title_text="Value",
)

fig.update_xaxes(
    axis_style,
    row=2,
    col=1,
    range=initial_abm_distribution["x_range"],
    title_text="Cumulative increment from origin, Xₜ − X₀",
)
fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    range=initial_abm_distribution["y_range"],
    title_text="Probability density",
)

fig.update_xaxes(
    axis_style,
    row=2,
    col=2,
    range=initial_gbm_distribution["x_range"],
    title_text="Gross return from origin, Sₜ / S₀",
)
fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    range=initial_gbm_distribution["y_range"],
    title_text="Probability density",
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Visualizing Time Varying Path Distributions

There's no better way to understand the difference than observing their return distributions over time

$$\text{Arithmetic Mean:} \quad \overline{x}_\text{arith} = \frac{1}{n} \sum_{i=1}^n x_i \qquad\qquad \text{Geometric Mean:} \quad \overline{x}_\text{geom} = \left( \prod_{i=1}^n x_i \right)^{1/n}$$

 $$\text{ABM:} \quad X_t \sim \mathcal{N}\left(X_0 + \mu t,\;\; \sigma^2 t\right) \qquad\qquad \text{GBM:} \quad S_t \sim \log\mathcal{N}\left(\log S_0 + \left(\mu - \frac{1}{2}\sigma^2\right)t,\;\; \sigma^2 t\right)$$

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statistics import NormalDist

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

YEARS = 15
STEPS_PER_YEAR = 12
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

MU = 0.30                        # 30% annual drift
SIGMA = 0.22                     # 22% annual volatility

N_SAMPLES = 40_000               # Monte Carlo observations per frame
N_BINS_ABM = 100
N_BINS_GBM = 120
PDF_POINTS = 800
TAIL_PROB = 0.002                # show theoretical 0.2%-99.8% support

FRAME_STRIDE = 3                 # quarterly frames
FRAME_DURATION = 65
INITIAL_I = 3                    # begin at 3 months

OUTPUT_HTML = "abm_vs_gbm_distribution_widening.html"
SHOW_FIG = True

if INITIAL_I < 1:
    raise ValueError("INITIAL_I must be at least 1 because t=0 is degenerate.")
if FRAME_STRIDE < 1:
    raise ValueError("FRAME_STRIDE must be at least 1.")

# ============================================================
# Distribution helpers
# ============================================================

def normal_pdf(x, mean, std):
    """Normal probability density."""
    x = np.asarray(x, dtype=float)
    z = (x - mean) / std
    return np.exp(-0.5 * z**2) / (std * np.sqrt(2 * np.pi))


def lognormal_pdf(x, log_mean, log_std):
    """
    Lognormal density for Y where:
        log(Y) ~ Normal(log_mean, log_std**2)
    """
    x = np.asarray(x, dtype=float)
    pdf = np.zeros_like(x)
    positive = x > 0

    z = (np.log(x[positive]) - log_mean) / log_std
    pdf[positive] = (
        np.exp(-0.5 * z**2)
        / (x[positive] * log_std * np.sqrt(2 * np.pi))
    )
    return pdf


def density_histogram(samples, bin_edges):
    """
    Return centers, widths, and probability-density heights.

    The full sample count stays in the denominator. Therefore, if a tiny
    fraction falls outside the fixed plotting support, the visible bar area
    is slightly below 1 instead of being renormalized upward.
    """
    counts, _ = np.histogram(samples, bins=bin_edges)
    widths = np.diff(bin_edges)
    centers = bin_edges[:-1] + 0.5 * widths
    density = counts / (samples.size * widths)
    return centers, widths, density


def padded_range(values, pad_fraction=0.08, min_pad=0.10):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))

    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)

    return [v_min - pad, v_max + pad]


def horizon_samples(t, z_abm, z_gbm):
    """
    Draw the horizon-from-origin distributions.

    ABM change:
        X_t - X_0 = mu*t + sigma*sqrt(t)*Z
                  ~ Normal(mu*t, sigma^2*t)

    GBM gross return:
        S_t / S_0 = exp((mu - 0.5*sigma^2)*t + sigma*sqrt(t)*Z)
                  ~ Lognormal(...)
    """
    root_t = np.sqrt(t)

    abm_change = MU * t + SIGMA * root_t * z_abm

    gbm_log_mean = (MU - 0.5 * SIGMA**2) * t
    gbm_log_std = SIGMA * root_t
    gbm_growth = np.exp(gbm_log_mean + gbm_log_std * z_gbm)

    return abm_change, gbm_growth


# ============================================================
# Time axis and frame horizons
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)
times = np.arange(N_STEPS + 1) * DT

frame_indices = list(range(INITIAL_I, N_STEPS + 1, FRAME_STRIDE))
if frame_indices[-1] != N_STEPS:
    frame_indices.append(N_STEPS)

frame_times = times[frame_indices]

# ============================================================
# Common random numbers
# ============================================================
#
# Reusing the same standard-normal draws makes the distributions morph
# smoothly. Each frame still has the exact required marginal law.
# ============================================================

z_abm = rng.normal(size=N_SAMPLES)
z_gbm = rng.normal(size=N_SAMPLES)

# ============================================================
# Fixed x-axis support and bins
# ============================================================

standard_normal = NormalDist()
z_lo = standard_normal.inv_cdf(TAIL_PROB)
z_hi = standard_normal.inv_cdf(1.0 - TAIL_PROB)

# ABM support: evaluate tail quantiles over every displayed horizon.
abm_q_lo = MU * frame_times + SIGMA * np.sqrt(frame_times) * z_lo
abm_q_hi = MU * frame_times + SIGMA * np.sqrt(frame_times) * z_hi

abm_x_range = padded_range(
    np.r_[0.0, abm_q_lo, abm_q_hi],
    pad_fraction=0.05,
    min_pad=0.10,
)
abm_bin_edges = np.linspace(
    abm_x_range[0],
    abm_x_range[1],
    N_BINS_ABM + 1,
)
abm_pdf_x = np.linspace(
    abm_x_range[0],
    abm_x_range[1],
    PDF_POINTS,
)

# GBM support: gross-return-factor tail quantiles.
gbm_log_means = (MU - 0.5 * SIGMA**2) * frame_times
gbm_log_stds = SIGMA * np.sqrt(frame_times)

gbm_q_lo = np.exp(gbm_log_means + gbm_log_stds * z_lo)
gbm_q_hi = np.exp(gbm_log_means + gbm_log_stds * z_hi)

gbm_x_min = max(
    float(np.min(gbm_q_lo)) * 0.92,
    np.finfo(float).tiny,
)
gbm_x_max = float(np.max(gbm_q_hi)) * 1.05
gbm_x_range = [gbm_x_min, gbm_x_max]

# Log-spaced bins give useful resolution near 1 while still covering
# the long right tail. The display axis itself remains linear.
gbm_bin_edges = np.geomspace(
    gbm_x_min,
    gbm_x_max,
    N_BINS_GBM + 1,
)
gbm_pdf_x = np.geomspace(
    gbm_x_min,
    gbm_x_max,
    PDF_POINTS,
)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

hist_color = "rgba(80,145,255,0.55)"
hist_line_color = "rgba(155,195,255,0.80)"
pdf_color = "#63d8ff"

abm_mean_color = "#ffaa33"
gbm_arith_color = "#00ff88"
gbm_geom_color = "#ffd84d"

baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Frame builder
# ============================================================

def build_frame(i):
    t = times[i]
    frame_date = dates[i]

    abm_change, gbm_growth = horizon_samples(
        t=t,
        z_abm=z_abm,
        z_gbm=z_gbm,
    )

    # ABM histogram and theoretical normal density
    abm_centers, abm_widths, abm_density = density_histogram(
        abm_change,
        abm_bin_edges,
    )
    abm_mean = MU * t
    abm_std = SIGMA * np.sqrt(t)
    abm_pdf_y = normal_pdf(abm_pdf_x, abm_mean, abm_std)

    abm_y_max = 1.10 * max(
        float(np.max(abm_density)),
        float(np.max(abm_pdf_y)),
    )

    # GBM histogram and theoretical lognormal density
    gbm_centers, gbm_widths, gbm_density = density_histogram(
        gbm_growth,
        gbm_bin_edges,
    )
    gbm_log_mean = (MU - 0.5 * SIGMA**2) * t
    gbm_log_std = SIGMA * np.sqrt(t)
    gbm_pdf_y = lognormal_pdf(
        gbm_pdf_x,
        gbm_log_mean,
        gbm_log_std,
    )

    gbm_arith_mean = np.exp(MU * t)
    gbm_geom_mean = np.exp(gbm_log_mean)
    gbm_y_max = 1.10 * max(
        float(np.max(gbm_density)),
        float(np.max(gbm_pdf_y)),
    )

    if i < STEPS_PER_YEAR:
        horizon_text = f"{i} months"
    else:
        horizon_text = f"{t:.2f} years"

    title_text = (
        "Distribution Widening from the Origin"
        f"<br><sup>Horizon: {horizon_text}"
        f" &nbsp;|&nbsp; Date: {frame_date:%Y-%m-%d}</sup>"
    )

    frame_data = [
        # 0: ABM Monte Carlo density
        go.Bar(
            x=abm_centers,
            y=abm_density,
            width=abm_widths,
            marker=dict(
                color=hist_color,
                line=dict(color=hist_line_color, width=0.5),
            ),
            opacity=0.80,
        ),

        # 1: ABM normal PDF
        go.Scatter(
            x=abm_pdf_x,
            y=abm_pdf_y,
            mode="lines",
            line=dict(color=pdf_color, width=4),
        ),

        # 2: ABM theoretical mean
        go.Scatter(
            x=[abm_mean, abm_mean],
            y=[0.0, abm_y_max],
            mode="lines",
            line=dict(
                color=abm_mean_color,
                width=3,
                dash="dash",
            ),
        ),

        # 3: GBM Monte Carlo density
        go.Bar(
            x=gbm_centers,
            y=gbm_density,
            width=gbm_widths,
            marker=dict(
                color=hist_color,
                line=dict(color=hist_line_color, width=0.5),
            ),
            opacity=0.80,
        ),

        # 4: GBM lognormal PDF
        go.Scatter(
            x=gbm_pdf_x,
            y=gbm_pdf_y,
            mode="lines",
            line=dict(color=pdf_color, width=4),
        ),

        # 5: GBM arithmetic mean
        go.Scatter(
            x=[gbm_arith_mean, gbm_arith_mean],
            y=[0.0, gbm_y_max],
            mode="lines",
            line=dict(color=gbm_arith_color, width=3),
        ),

        # 6: GBM geometric mean / median
        go.Scatter(
            x=[gbm_geom_mean, gbm_geom_mean],
            y=[0.0, gbm_y_max],
            mode="lines",
            line=dict(
                color=gbm_geom_color,
                width=3,
                dash="dash",
            ),
        ),
    ]

    frame_layout = go.Layout(
        title=dict(
            text=title_text,
            x=0.5,
            font=dict(color=off_white),
        ),
        yaxis=dict(range=[0.0, abm_y_max]),
        yaxis2=dict(range=[0.0, gbm_y_max]),
    )

    return frame_data, frame_layout


# ============================================================
# Initial figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.08,
    subplot_titles=(
        "ABM: Change from Origin is Normal",
        "GBM: Growth Factor is Lognormal",
    ),
)

initial_i = frame_indices[0]
initial_data, initial_layout = build_frame(initial_i)

initial_data[0].update(
    name="Monte Carlo density",
    legendgroup="histogram",
    showlegend=True,
    hovertemplate=(
        "ABM change bin center: %{x:.4f}<br>"
        "Estimated density: %{y:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[0], row=1, col=1)

initial_data[1].update(
    name="Theoretical PDF",
    legendgroup="pdf",
    showlegend=True,
    hovertemplate=(
        "ABM change: %{x:.4f}<br>"
        "Normal PDF: %{y:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[1], row=1, col=1)

initial_data[2].update(
    name="ABM mean",
    legendgroup="abm-mean",
    showlegend=True,
    hovertemplate="ABM mean change: %{x:.4f}<extra></extra>",
)
fig.add_trace(initial_data[2], row=1, col=1)

initial_data[3].update(
    name="Monte Carlo density",
    legendgroup="histogram",
    showlegend=False,
    hovertemplate=(
        "GBM growth-factor bin center: %{x:.4f}<br>"
        "Estimated density: %{y:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[3], row=1, col=2)

initial_data[4].update(
    name="Theoretical PDF",
    legendgroup="pdf",
    showlegend=False,
    hovertemplate=(
        "Growth factor: %{x:.4f}<br>"
        "Lognormal PDF: %{y:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[4], row=1, col=2)

initial_data[5].update(
    name="GBM arithmetic mean",
    legendgroup="gbm-arith-mean",
    showlegend=True,
    hovertemplate="E[Sₜ / S₀]: %{x:.4f}<extra></extra>",
)
fig.add_trace(initial_data[5], row=1, col=2)

initial_data[6].update(
    name="GBM geometric mean / median",
    legendgroup="gbm-geom-mean",
    showlegend=True,
    hovertemplate=(
        "exp(E[log(Sₜ / S₀)]): %{x:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[6], row=1, col=2)

# Origin baselines
fig.add_vline(
    x=0.0,
    line=dict(color=baseline_color, width=1, dash="dot"),
    opacity=0.75,
    row=1,
    col=1,
)

fig.add_vline(
    x=1.0,
    line=dict(color=baseline_color, width=1, dash="dot"),
    opacity=0.75,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []
animated_trace_indices = list(range(7))

for i in frame_indices:
    frame_name = f"f{i}"
    frame_data, frame_layout = build_frame(i)

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=animated_trace_indices,
            layout=frame_layout,
        )
    )

    elapsed_years = i / STEPS_PER_YEAR
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {
                    "duration": 0,
                    "redraw": False,
                },
                "transition": {"duration": 0},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=initial_layout.title,
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700,
    width=1280,
    margin=dict(t=110, b=130, r=40, l=75),
    barmode="overlay",
    bargap=0.0,
    legend=dict(
        orientation="v",
        x=0.0,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False,
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False,
                        },
                        "transition": {"duration": 0},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 70},
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": 0.0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {"visible": False},
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 40},
        "len": 0.85,
        "x": 0.15,
        "y": 0.0,
        "steps": slider_steps,
    }],
)

fig.update_annotations(font=dict(color=off_white, size=16))

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=abm_x_range,
    title_text="Change from Origin: Xₜ − X₀",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=initial_layout.yaxis.range,
    title_text="Probability Density",
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=gbm_x_range,
    title_text="Growth Factor from Origin: Sₜ / S₀",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=initial_layout.yaxis2.range,
    title_text="Probability Density",
)


fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Visualizing Time Varying Increments in the Stochastic Path
 
 Let's focus on the increment (change in one step) for each process.
 
 $$\text{ABM:} \quad \Delta X = X_{t+\Delta t} - X_t = \mu \Delta t + \sigma \sqrt{\Delta t} \cdot Z \implies \text{Normal Distribution}$$
 
 $$\text{GBM:} \quad S_{t+\Delta t} = S_t \cdot \exp\left[\left(\mu - \frac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t}\cdot Z \right] \implies \text{Scaled and Shifted Lognormals}$$
 
 

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from numpy.polynomial.hermite import hermgauss
from plotly.subplots import make_subplots
from statistics import NormalDist

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

YEARS = 15
STEPS_PER_YEAR = 12              # monthly steps
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

MU = 0.30                        # 30% annual drift
SIGMA = 0.22                     # 22% annual volatility

N_SAMPLES = 40_000               # Monte Carlo observations per frame
N_BINS_ABM = 110
N_BINS_GBM = 150
PDF_POINTS = 900
TAIL_PROB = 0.002                # display central 99.6% of each density
GH_NODES = 70                    # Gauss-Hermite nodes for exact GBM mixture PDF

FRAME_STRIDE = 3                 # quarterly frames after the first month
FRAME_DURATION = 65

OUTPUT_HTML = "abm_vs_gbm_increment_distributions.html"
SHOW_FIG = True

if FRAME_STRIDE < 1:
    raise ValueError("FRAME_STRIDE must be at least 1.")
if N_STEPS < 1:
    raise ValueError("N_STEPS must be at least 1.")

# ============================================================
# Distribution helpers
# ============================================================


def normal_pdf(x, mean, std):
    """Normal probability density."""
    x = np.asarray(x, dtype=float)
    z = (x - mean) / std
    return np.exp(-0.5 * z**2) / (std * np.sqrt(2.0 * np.pi))


def density_histogram(samples, bin_edges):
    """
    Return bin centers, widths, and probability-density heights.

    The full sample count remains in the denominator. Consequently, if the
    displayed bins omit tiny tails, their visible area is slightly below one
    rather than being renormalized upward.
    """
    samples = np.asarray(samples, dtype=float)
    bin_edges = np.asarray(bin_edges, dtype=float)

    counts, _ = np.histogram(samples, bins=bin_edges)
    widths = np.diff(bin_edges)
    centers = bin_edges[:-1] + 0.5 * widths
    density = counts / (samples.size * widths)
    return centers, widths, density


def padded_range(values, pad_fraction=0.08, min_pad=0.01):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))

    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)

    return [v_min - pad, v_max + pad]


def central_sample_range(samples, tail_prob=TAIL_PROB, pad_fraction=0.04):
    """Frame-specific central range used for bins and PDF evaluation."""
    q_lo, q_hi = np.quantile(samples, [tail_prob, 1.0 - tail_prob])
    return padded_range(
        [q_lo, q_hi],
        pad_fraction=pad_fraction,
        min_pad=1e-4,
    )


def format_horizon(step_index):
    """Compact slider label for elapsed time from the origin."""
    if step_index < STEPS_PER_YEAR:
        return f"{step_index}M"
    years = step_index / STEPS_PER_YEAR
    if np.isclose(years, round(years)):
        return f"{int(round(years))}Y"
    return f"{years:.1f}Y"


# ============================================================
# Model-specific increment laws
# ============================================================


def simulate_abm_increment(z):
    """
    One monthly ABM increment:

        Delta X = mu*DT + sigma*sqrt(DT)*Z

    Its distribution is identical at every point in calendar time.
    """
    return MU * DT + SIGMA * np.sqrt(DT) * z


def simulate_gbm_increment(t_previous, z_level, z_step):
    """
    One monthly GBM dollar increment at elapsed time t_previous + DT.

    The prior level and next gross return are independent:

        S_previous = S0 * exp((mu - 0.5*sigma^2)t_previous
                              + sigma*sqrt(t_previous)*Z_level)

        R_step = exp((mu - 0.5*sigma^2)DT
                     + sigma*sqrt(DT)*Z_step)

        Delta S = S_previous * (R_step - 1)

    The percentage increment R_step - 1 has the same law every month, but the
    dollar increment Delta S is state dependent because it is scaled by the
    random prior level S_previous.
    """
    if t_previous <= 0.0:
        previous_level = np.full_like(z_level, INITIAL_VALUE, dtype=float)
    else:
        previous_level = INITIAL_VALUE * np.exp(
            (MU - 0.5 * SIGMA**2) * t_previous
            + SIGMA * np.sqrt(t_previous) * z_level
        )

    step_growth = np.exp(
        (MU - 0.5 * SIGMA**2) * DT
        + SIGMA * np.sqrt(DT) * z_step
    )

    return previous_level * (step_growth - 1.0)


def gbm_increment_theoretical_pdf(x, t_previous, gh_nodes, gh_weights):
    """
    Exact unconditional PDF of the one-step GBM dollar increment.

    Conditional on S_previous=s,

        Delta S = s * (R_step - 1),

    where R_step is lognormal. Thus the conditional increment density is a
    shifted-and-scaled lognormal density. The unconditional density is the
    mixture over the lognormal prior-level distribution:

        f_DeltaS(y) = E[ f_R(1 + y/S_previous) / S_previous ].

    The expectation is evaluated deterministically with Gauss-Hermite
    quadrature. This is a theoretical mixture PDF, not a KDE or normal fit.
    """
    x = np.asarray(x, dtype=float)

    step_log_mean = (MU - 0.5 * SIGMA**2) * DT
    step_log_std = SIGMA * np.sqrt(DT)

    if t_previous <= 0.0:
        levels = np.array([INITIAL_VALUE], dtype=float)
        weights = np.array([1.0], dtype=float)
    else:
        previous_log_mean = (MU - 0.5 * SIGMA**2) * t_previous
        previous_log_std = SIGMA * np.sqrt(t_previous)

        standard_normal_nodes = np.sqrt(2.0) * gh_nodes
        levels = INITIAL_VALUE * np.exp(
            previous_log_mean
            + previous_log_std * standard_normal_nodes
        )
        weights = gh_weights / np.sqrt(np.pi)

    # For each prior level s, r = 1 + y/s must be positive.
    ratio = 1.0 + x[:, None] / levels[None, :]
    valid = ratio > 0.0

    log_density = np.full_like(ratio, -np.inf, dtype=float)
    log_ratio = np.zeros_like(ratio, dtype=float)
    log_ratio[valid] = np.log(ratio[valid])

    z = np.zeros_like(ratio, dtype=float)
    z[valid] = (
        log_ratio[valid] - step_log_mean
    ) / step_log_std

    log_density[valid] = (
        -0.5 * z[valid] ** 2
        - log_ratio[valid]
        - np.log(step_log_std * np.sqrt(2.0 * np.pi))
        - np.log(np.broadcast_to(levels[None, :], ratio.shape)[valid])
    )

    conditional_density = np.exp(log_density)
    return conditional_density @ weights


def gbm_expected_increment(t_previous):
    """Exact E[Delta S] for the one-step interval after t_previous."""
    expected_previous_level = INITIAL_VALUE * np.exp(MU * t_previous)
    expected_step_growth_minus_one = np.exp(MU * DT) - 1.0
    return expected_previous_level * expected_step_growth_minus_one


# ============================================================
# Time axis and animation frames
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)
times = np.arange(N_STEPS + 1) * DT

# Include the first monthly increment, then reveal quarterly.
frame_indices = [1]
frame_indices.extend(range(FRAME_STRIDE, N_STEPS + 1, FRAME_STRIDE))
frame_indices = sorted(set(frame_indices))
if frame_indices[-1] != N_STEPS:
    frame_indices.append(N_STEPS)

# ============================================================
# Common random numbers
# ============================================================
#
# ABM uses one fixed sample because its one-step law never changes.
# GBM reuses common draws across frames so its state-dependent density morphs
# smoothly rather than flickering because of fresh Monte Carlo noise.
# ============================================================

z_abm = rng.normal(size=N_SAMPLES)
z_level = rng.normal(size=N_SAMPLES)
z_step = rng.normal(size=N_SAMPLES)

abm_increment_samples = simulate_abm_increment(z_abm)

# Gauss-Hermite quadrature nodes for the exact GBM mixture density.
gh_nodes, gh_weights = hermgauss(GH_NODES)

# ============================================================
# Fixed display support
# ============================================================

standard_normal = NormalDist()
z_lo = standard_normal.inv_cdf(TAIL_PROB)
z_hi = standard_normal.inv_cdf(1.0 - TAIL_PROB)

# ABM support is constant because the increment law is time invariant.
abm_mean_increment = MU * DT
abm_std_increment = SIGMA * np.sqrt(DT)
abm_q_lo = abm_mean_increment + abm_std_increment * z_lo
abm_q_hi = abm_mean_increment + abm_std_increment * z_hi
abm_x_range = padded_range(
    [0.0, abm_q_lo, abm_q_hi],
    pad_fraction=0.08,
    min_pad=0.015,
)
abm_bin_edges = np.linspace(
    abm_x_range[0],
    abm_x_range[1],
    N_BINS_ABM + 1,
)
abm_pdf_x = np.linspace(
    abm_x_range[0],
    abm_x_range[1],
    PDF_POINTS,
)

# Cache each GBM frame's Monte Carlo increment sample and its local central
# range. The final right-hand x-axis spans every displayed frame, making the
# state-dependent widening directly visible.
gbm_samples_by_step = {}
gbm_local_ranges = {}
gbm_global_bounds = [0.0]

for i in frame_indices:
    t_previous = times[i - 1]
    samples = simulate_gbm_increment(
        t_previous=t_previous,
        z_level=z_level,
        z_step=z_step,
    )
    local_range = central_sample_range(samples)

    gbm_samples_by_step[i] = samples
    gbm_local_ranges[i] = local_range
    gbm_global_bounds.extend(local_range)

gbm_x_range = padded_range(
    gbm_global_bounds,
    pad_fraction=0.035,
    min_pad=0.03,
)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

hist_color = "rgba(80,145,255,0.55)"
hist_line_color = "rgba(155,195,255,0.80)"
pdf_color = "#63d8ff"

abm_mean_color = "#ffaa33"
gbm_mean_color = "#00ff88"
zero_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Frame builder
# ============================================================


def build_frame(i):
    t_previous = times[i - 1]
    t_current = times[i]

    # --------------------------------------------------------
    # ABM: identical one-step normal distribution every frame
    # --------------------------------------------------------
    abm_centers, abm_widths, abm_density = density_histogram(
        abm_increment_samples,
        abm_bin_edges,
    )
    abm_pdf_y = normal_pdf(
        abm_pdf_x,
        abm_mean_increment,
        abm_std_increment,
    )
    abm_y_max = 1.10 * max(
        float(np.max(abm_density)),
        float(np.max(abm_pdf_y)),
    )

    # --------------------------------------------------------
    # GBM: state-dependent one-step dollar increment
    # --------------------------------------------------------
    gbm_samples = gbm_samples_by_step[i]
    gbm_local_range = gbm_local_ranges[i]

    # Frame-specific bins preserve detail at early dates while the display
    # axis stays fixed across the complete animation.
    gbm_bin_edges = np.linspace(
        gbm_local_range[0],
        gbm_local_range[1],
        N_BINS_GBM + 1,
    )
    gbm_pdf_x = np.linspace(
        gbm_local_range[0],
        gbm_local_range[1],
        PDF_POINTS,
    )

    gbm_centers, gbm_widths, gbm_density = density_histogram(
        gbm_samples,
        gbm_bin_edges,
    )
    gbm_pdf_y = gbm_increment_theoretical_pdf(
        gbm_pdf_x,
        t_previous=t_previous,
        gh_nodes=gh_nodes,
        gh_weights=gh_weights,
    )
    gbm_mean_increment = gbm_expected_increment(t_previous)
    gbm_y_max = 1.10 * max(
        float(np.max(gbm_density)),
        float(np.max(gbm_pdf_y)),
    )

    title_text = (
        "One-Step Increment Distributions Over Time"
        f"<br><sup>Monthly increment: {dates[i - 1]:%Y-%m-%d}"
        f" → {dates[i]:%Y-%m-%d}"
        f" &nbsp;|&nbsp; Elapsed horizon: {format_horizon(i)}</sup>"
    )

    frame_data = [
        # 0: ABM Monte Carlo density
        go.Bar(
            x=abm_centers,
            y=abm_density,
            width=abm_widths,
            marker=dict(
                color=hist_color,
                line=dict(color=hist_line_color, width=0.5),
            ),
            opacity=0.80,
        ),

        # 1: ABM exact normal PDF
        go.Scatter(
            x=abm_pdf_x,
            y=abm_pdf_y,
            mode="lines",
            line=dict(color=pdf_color, width=4),
        ),

        # 2: ABM expected increment
        go.Scatter(
            x=[abm_mean_increment, abm_mean_increment],
            y=[0.0, abm_y_max],
            mode="lines",
            line=dict(
                color=abm_mean_color,
                width=3,
                dash="dash",
            ),
        ),

        # 3: GBM Monte Carlo density
        go.Bar(
            x=gbm_centers,
            y=gbm_density,
            width=gbm_widths,
            marker=dict(
                color=hist_color,
                line=dict(color=hist_line_color, width=0.5),
            ),
            opacity=0.80,
        ),

        # 4: GBM exact state-dependent mixture PDF
        go.Scatter(
            x=gbm_pdf_x,
            y=gbm_pdf_y,
            mode="lines",
            line=dict(color=pdf_color, width=4),
        ),

        # 5: GBM expected dollar increment
        go.Scatter(
            x=[gbm_mean_increment, gbm_mean_increment],
            y=[0.0, gbm_y_max],
            mode="lines",
            line=dict(color=gbm_mean_color, width=3),
        ),
    ]

    frame_layout = go.Layout(
        title=dict(
            text=title_text,
            x=0.5,
            font=dict(color=off_white),
        ),
        yaxis=dict(range=[0.0, abm_y_max]),
        yaxis2=dict(range=[0.0, gbm_y_max]),
    )

    return frame_data, frame_layout


# ============================================================
# Initial figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.08,
    subplot_titles=(
        "ABM: Increment Law is Fixed",
        "GBM: Dollar Increment is State Dependent",
    ),
)

initial_i = frame_indices[0]
initial_data, initial_layout = build_frame(initial_i)

initial_data[0].update(
    name="Monte Carlo density",
    legendgroup="histogram",
    showlegend=True,
    hovertemplate=(
        "ABM increment bin center: %{x:.5f}<br>"
        "Estimated density: %{y:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[0], row=1, col=1)

initial_data[1].update(
    name="ABM normal PDF",
    legendgroup="abm-pdf",
    showlegend=True,
    hovertemplate=(
        "ABM increment: %{x:.5f}<br>"
        "Normal PDF: %{y:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[1], row=1, col=1)

initial_data[2].update(
    name="ABM expected increment",
    legendgroup="abm-mean",
    showlegend=True,
    hovertemplate="E[Delta X]: %{x:.5f}<extra></extra>",
)
fig.add_trace(initial_data[2], row=1, col=1)

initial_data[3].update(
    name="Monte Carlo density",
    legendgroup="histogram",
    showlegend=False,
    hovertemplate=(
        "GBM dollar-increment bin center: %{x:.5f}<br>"
        "Estimated density: %{y:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[3], row=1, col=2)

initial_data[4].update(
    name="GBM theoretical mixture PDF",
    legendgroup="gbm-pdf",
    showlegend=True,
    hovertemplate=(
        "GBM dollar increment: %{x:.5f}<br>"
        "Mixture PDF: %{y:.4f}<extra></extra>"
    ),
)
fig.add_trace(initial_data[4], row=1, col=2)

initial_data[5].update(
    name="GBM expected increment",
    legendgroup="gbm-mean",
    showlegend=True,
    hovertemplate="E[Delta S]: %{x:.5f}<extra></extra>",
)
fig.add_trace(initial_data[5], row=1, col=2)

# Zero-increment baselines
fig.add_vline(
    x=0.0,
    line=dict(color=zero_color, width=1, dash="dot"),
    opacity=0.75,
    row=1,
    col=1,
)
fig.add_vline(
    x=0.0,
    line=dict(color=zero_color, width=1, dash="dot"),
    opacity=0.75,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []
animated_trace_indices = list(range(6))

for i in frame_indices:
    frame_name = f"f{i}"
    frame_data, frame_layout = build_frame(i)

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=animated_trace_indices,
            layout=frame_layout,
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {
                    "duration": 0,
                    "redraw": False,
                },
                "transition": {"duration": 0},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": format_horizon(i),
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=initial_layout.title,
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700,
    width=1280,
    margin=dict(t=110, b=130, r=40, l=75),
    barmode="overlay",
    bargap=0.0,
    legend=dict(
        orientation="v",
        x=0.0,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False,
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False,
                        },
                        "transition": {"duration": 0},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 70},
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": 0.0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {"visible": False},
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 40},
        "len": 0.85,
        "x": 0.15,
        "y": 0.0,
        "steps": slider_steps,
    }],
)

fig.update_annotations(font=dict(color=off_white, size=16))

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=abm_x_range,
    title_text="Monthly Increment: Delta X",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=initial_layout.yaxis.range,
    title_text="Probability Density",
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=gbm_x_range,
    title_text="Monthly Dollar Increment: Delta S",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=initial_layout.yaxis2.range,
    title_text="Probability Density",
)


---

##### Arithmetic Brownian Motion

Step-by-step explicit solution to Arithmetic Brownian Motion (ABM):

The SDE for ABM is:
$$
dX_t = \mu\,dt + \sigma\,dW_t
$$

1. Integrate both sides from 0 to $t$:
$$
\int_0^t dX_s = \int_0^t \mu\,ds + \int_0^t \sigma\,dW_s
$$

2. This gives:
$$
X_t - X_0 = \mu t + \sigma (W_t - W_0)
$$

3. Since $W_0 = 0$:
$$
X_t = X_0 + \mu t + \sigma W_t
$$

4. Therefore, $X_t$ is normally distributed:
$$
X_t \sim \mathcal{N}(X_0 + \mu t,\, \sigma^2 t)
$$

Alternatively, for simulation:
$$
X_t = X_0 + \mu t + \sigma \sqrt{t}\,Z
$$
where $Z \sim \mathcal{N}(0,1)$.

###### ______________________________________________________________________________________________________________________________________

##### Convergence to the Theoretical Distribution

$X_t \sim \mathcal{N}(X_0 + \mu t,\, \sigma^2 t)$

By the Law of Large Numbers, empirical probabilities, distributions, and statistics of $X_t$ converge to their theoretical (normal) values as the number of samples increases.

 By the Law of Large Numbers, as the number of simulated paths $n \to \infty$:
 $$
 \bar{X}_t^{(n)} = \frac{1}{n} \sum_{i=1}^n X_t^{(i)} \xrightarrow{P} \mathbb{E}[X_t] = X_0 + \mu t
 $$
 $$
 S^2_t = \frac{1}{n} \sum_{i=1}^n \left(X_t^{(i)} - \bar{X}_t^{(n)}\right)^2 \xrightarrow{P} \operatorname{Var}[X_t] = \sigma^2 t
 $$
 where $\xrightarrow{P}$ denotes convergence in probability.

In [ ]:
import math
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

YEARS = 15
STEPS_PER_YEAR = 12              # monthly steps
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

# More than 30 paths makes convergence of the empirical terminal
# histogram much easier to see. Each animation frame adds one path.
N_PATHS = 200
INITIAL_PATHS = 10

MU = 0.30                        # 30% annual drift
SIGMA = 0.22                     # 22% annual volatility

HIST_BINS = 28
PDF_POINTS = 500
TERMINAL_STD_SPAN = 4.25

FRAME_DURATION = 65
OUTPUT_HTML = "abm_terminal_distribution_convergence.html"

# Use SHOW_FIG=0 in the environment to suppress fig.show().
SHOW_FIG = os.getenv("SHOW_FIG", "1") == "1"

# ============================================================
# Simulation and distribution helpers
# ============================================================


def simulate_abm_paths(x0, mu, sigma, n_steps, n_paths, dt, rng):
    """
    Simulate Arithmetic Brownian Motion paths:

        dX_t = mu dt + sigma dW_t

    Returns an array shaped (n_steps + 1, n_paths).
    """
    z = rng.normal(size=(n_steps, n_paths))
    increments = mu * dt + sigma * np.sqrt(dt) * z

    return x0 + np.vstack([
        np.zeros(n_paths),
        np.cumsum(increments, axis=0),
    ])


def abm_theoretical_mean(x0, mu, times):
    return x0 + mu * times


def normal_pdf(x, mean, std):
    x = np.asarray(x, dtype=float)
    z = (x - mean) / std
    return np.exp(-0.5 * z**2) / (std * np.sqrt(2.0 * np.pi))


def normal_cdf(x, mean, std):
    """Vectorized normal CDF without requiring SciPy."""
    x = np.asarray(x, dtype=float)
    scale = std * np.sqrt(2.0)
    return np.fromiter(
        (0.5 * (1.0 + math.erf((value - mean) / scale)) for value in x),
        dtype=float,
        count=x.size,
    )


def empirical_density(values, bin_edges):
    """Return histogram density heights using fixed bin edges."""
    counts, _ = np.histogram(values, bins=bin_edges)
    bin_widths = np.diff(bin_edges)
    return counts / (len(values) * bin_widths)


def separated_paths(x_values, path_matrix, path_indices):
    """
    Combine multiple paths into a single Scatter trace using NaN separators.

    This helper is retained for experimentation, although the animation below
    uses one base trace per path so slider jumps can reveal the exact path set.
    """
    if len(path_indices) == 0:
        return np.array([], dtype=object), np.array([], dtype=float)

    x_parts = []
    y_parts = []

    for j in path_indices:
        x_parts.extend(x_values)
        x_parts.append(None)
        y_parts.extend(path_matrix[:, j])
        y_parts.append(np.nan)

    return np.asarray(x_parts, dtype=object), np.asarray(y_parts, dtype=float)


def padded_range(values, pad_fraction=0.08, min_pad=0.10):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))

    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)

    return [v_min - pad, v_max + pad]


# ============================================================
# Time axis and ABM simulation
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)

times = np.arange(N_STEPS + 1) * DT
terminal_time = YEARS
terminal_date = dates[-1]

abm_paths = simulate_abm_paths(
    x0=INITIAL_VALUE,
    mu=MU,
    sigma=SIGMA,
    n_steps=N_STEPS,
    n_paths=N_PATHS,
    dt=DT,
    rng=rng,
)

abm_mean = abm_theoretical_mean(INITIAL_VALUE, MU, times)
terminal_values = abm_paths[-1, :]

# For ABM:
#     X_T ~ Normal(x0 + mu T, sigma^2 T)
terminal_theoretical_mean = INITIAL_VALUE + MU * terminal_time
terminal_theoretical_std = SIGMA * np.sqrt(terminal_time)

# ============================================================
# Fixed histogram and PDF support
# ============================================================

support_min = min(
    float(np.min(terminal_values)),
    terminal_theoretical_mean - TERMINAL_STD_SPAN * terminal_theoretical_std,
)

support_max = max(
    float(np.max(terminal_values)),
    terminal_theoretical_mean + TERMINAL_STD_SPAN * terminal_theoretical_std,
)

hist_edges = np.linspace(support_min, support_max, HIST_BINS + 1)
hist_centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])
hist_widths = np.diff(hist_edges)

pdf_x = np.linspace(support_min, support_max, PDF_POINTS)
pdf_y = normal_pdf(
    pdf_x,
    terminal_theoretical_mean,
    terminal_theoretical_std,
)

# Expected theoretical density in each histogram bin:
# P(bin) / bin width.
theoretical_bin_probabilities = np.diff(
    normal_cdf(
        hist_edges,
        terminal_theoretical_mean,
        terminal_theoretical_std,
    )
)
theoretical_bin_density = theoretical_bin_probabilities / hist_widths

# Calculate a fixed right-panel y-range that accommodates every animation frame.
all_empirical_densities = []
for n in range(INITIAL_PATHS, N_PATHS + 1):
    all_empirical_densities.append(
        empirical_density(terminal_values[:n], hist_edges)
    )

maximum_empirical_density = max(
    float(np.max(values)) for values in all_empirical_densities
)

right_y_max = 1.12 * max(
    maximum_empirical_density,
    float(np.max(theoretical_bin_density)),
    float(np.max(pdf_y)),
)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

abm_mean_color = "#ffaa33"       # orange
pdf_color = "#ffd84d"            # yellow
empirical_hist_color = "#4ea3ff" # blue
empirical_mean_color = "#00ff88" # green

path_color_over = "#18d618"      # green
path_color_under = "#ff3030"     # red

baseline_color = "#777777"
newest_terminal_color = "#ffffff"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.55, 0.45],
    horizontal_spacing=0.09,
    subplot_titles=(
        "ABM Sample Paths Added One at a Time",
        f"Terminal Distribution at T = {YEARS} Years",
    ),
)

# ------------------------------------------------------------
# Left panel: all complete paths, initially hidden except first n
# ------------------------------------------------------------

for j in range(N_PATHS):
    terminal_is_over_mean = terminal_values[j] > terminal_theoretical_mean
    path_color = path_color_over if terminal_is_over_mean else path_color_under

    fig.add_trace(
        go.Scatter(
            x=dates,
            y=abm_paths[:, j],
            mode="lines",
            line=dict(color=path_color, width=1.35),
            opacity=0.34,
            visible=(j < INITIAL_PATHS),
            name=f"{N_PATHS} ABM sample paths" if j == 0 else f"Path {j + 1}",
            legendgroup="abm-paths",
            showlegend=(j == 0),
            hovertemplate=(
                f"ABM path {j + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# Theoretical mean path is fixed throughout the animation.
fig.add_trace(
    go.Scatter(
        x=dates,
        y=abm_mean,
        mode="lines",
        line=dict(color=abm_mean_color, width=4),
        name="Theoretical mean path",
        legendgroup="abm-mean",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "E[X_t]: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)
mean_path_trace_index = len(fig.data) - 1

# A bright overlay highlights the newest complete path added by the frame.
initial_newest_index = INITIAL_PATHS - 1
initial_newest_color = (
    path_color_over
    if terminal_values[initial_newest_index] > terminal_theoretical_mean
    else path_color_under
)

fig.add_trace(
    go.Scatter(
        x=dates,
        y=abm_paths[:, initial_newest_index],
        mode="lines",
        line=dict(color=initial_newest_color, width=4),
        opacity=0.95,
        name="Newest path",
        legendgroup="newest-path",
        showlegend=True,
        hovertemplate=(
            f"Newest path: {initial_newest_index + 1}<br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Value: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)
newest_path_trace_index = len(fig.data) - 1

# ------------------------------------------------------------
# Right panel: fixed theoretical distribution and growing sample
# ------------------------------------------------------------

initial_terminal_sample = terminal_values[:INITIAL_PATHS]
initial_empirical_density = empirical_density(
    initial_terminal_sample,
    hist_edges,
)

# Theoretical histogram bin heights.
fig.add_trace(
    go.Bar(
        x=hist_centers,
        y=theoretical_bin_density,
        width=hist_widths,
        marker=dict(
            color="rgba(255,216,77,0.18)",
            line=dict(color=pdf_color, width=1.2),
        ),
        name="Theoretical bin density",
        legendgroup="theoretical-distribution",
        hovertemplate=(
            "Terminal value: %{x:.4f}<br>"
            "Theoretical bin density: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)
theoretical_hist_trace_index = len(fig.data) - 1

# Empirical histogram, updated every frame.
fig.add_trace(
    go.Bar(
        x=hist_centers,
        y=initial_empirical_density,
        width=hist_widths * 0.92,
        marker=dict(
            color="rgba(78,163,255,0.68)",
            line=dict(color=empirical_hist_color, width=1.0),
        ),
        name="Empirical terminal density",
        legendgroup="empirical-distribution",
        hovertemplate=(
            "Terminal value: %{x:.4f}<br>"
            "Empirical density: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)
empirical_hist_trace_index = len(fig.data) - 1

# Smooth theoretical PDF.
fig.add_trace(
    go.Scatter(
        x=pdf_x,
        y=pdf_y,
        mode="lines",
        line=dict(color=pdf_color, width=4),
        name="Theoretical normal PDF",
        legendgroup="theoretical-distribution",
        hovertemplate=(
            "Terminal value: %{x:.4f}<br>"
            "Normal PDF: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)
theoretical_pdf_trace_index = len(fig.data) - 1

# Empirical mean marker, updated every frame.
initial_empirical_mean = float(np.mean(initial_terminal_sample))
fig.add_trace(
    go.Scatter(
        x=[initial_empirical_mean, initial_empirical_mean],
        y=[0.0, right_y_max],
        mode="lines",
        line=dict(color=empirical_mean_color, width=2.5, dash="dash"),
        name="Empirical terminal mean",
        legendgroup="empirical-mean",
        hovertemplate=(
            "Empirical mean: %{x:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)
empirical_mean_trace_index = len(fig.data) - 1

# Latest endpoint marker along the baseline.
initial_latest_terminal = float(terminal_values[initial_newest_index])
fig.add_trace(
    go.Scatter(
        x=[initial_latest_terminal],
        y=[0.025 * right_y_max],
        mode="markers",
        marker=dict(
            color=newest_terminal_color,
            size=11,
            symbol="diamond",
            line=dict(color="rgba(0,0,0,0.75)", width=1),
        ),
        name="Newest terminal observation",
        legendgroup="newest-terminal",
        showlegend=False,
        hovertemplate=(
            "Newest terminal observation: %{x:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)
latest_terminal_trace_index = len(fig.data) - 1

# REMOVE: Dynamic summary text in the upper-left of the right panel.
# (No text overlay. Remove stats_trace_index etc.)

# Baseline and terminal markers.
fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1,
)

fig.add_vline(
    x=terminal_date,
    line=dict(color=baseline_color, width=1.2, dash="dash"),
    opacity=0.75,
    row=1,
    col=1,
)

fig.add_vline(
    x=terminal_theoretical_mean,
    line=dict(color=abm_mean_color, width=2.2, dash="dot"),
    opacity=0.95,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

base_path_trace_indices = list(range(N_PATHS))
# Remove stats_trace_index from animated_trace_indices
animated_trace_indices = base_path_trace_indices + [
    newest_path_trace_index,
    empirical_hist_trace_index,
    empirical_mean_trace_index,
    latest_terminal_trace_index,
    # stats_trace_index removed
]

for n in range(INITIAL_PATHS, N_PATHS + 1):
    newest_index = n - 1
    sample = terminal_values[:n]

    density = empirical_density(sample, hist_edges)
    sample_mean = float(np.mean(sample))
    sample_std = float(np.std(sample, ddof=1)) if n > 1 else 0.0
    newest_terminal = float(terminal_values[newest_index])

    newest_color = (
        path_color_over
        if newest_terminal > terminal_theoretical_mean
        else path_color_under
    )

    frame_data = []

    # Reveal exactly the first n complete paths. Including every visibility
    # state makes slider jumps deterministic, not dependent on playback history.
    for j in range(N_PATHS):
        frame_data.append(
            go.Scatter(visible=(j < n))
        )

    # Bright newest-path overlay.
    frame_data.append(
        go.Scatter(
            x=dates,
            y=abm_paths[:, newest_index],
            mode="lines",
            line=dict(color=newest_color, width=4),
            opacity=0.95,
            hovertemplate=(
                f"Newest path: {newest_index + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        )
    )

    # Empirical terminal histogram.
    frame_data.append(
        go.Bar(
            x=hist_centers,
            y=density,
            width=hist_widths * 0.92,
        )
    )

    # Empirical terminal mean.
    frame_data.append(
        go.Scatter(
            x=[sample_mean, sample_mean],
            y=[0.0, right_y_max],
            mode="lines",
        )
    )

    # Latest endpoint marker.
    frame_data.append(
        go.Scatter(
            x=[newest_terminal],
            y=[0.025 * right_y_max],
            mode="markers",
        )
    )

    # Remove dynamic stats text trace.

    frame_name = f"paths_{n}"

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=animated_trace_indices,
        )
    )

    slider_label = str(n) if (
        n == INITIAL_PATHS
        or n == N_PATHS
        or n % 10 == 0
    ) else ""

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "transition": {"duration": 0},
            },
        ],
        "label": slider_label,
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Axis ranges
# ============================================================

left_values = np.r_[abm_paths.ravel(), abm_mean]
left_y_range = padded_range(
    left_values,
    pad_fraction=0.07,
    min_pad=0.20,
)

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "ABM Monte Carlo Convergence: "
            "Complete Paths to the Terminal Normal Distribution"
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=720,
    width=1320,
    margin=dict(t=105, b=135, r=45, l=70),
    barmode="overlay",
    bargap=0.02,
    legend=dict(
        orientation="v",
        x=0.0,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.78)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False,
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                        "mode": "immediate",
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "transition": {"duration": 0},
                        "mode": "immediate",
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 75},
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": 0.0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "visible": True,
            "prefix": "Paths included: ",
            "font": {"color": off_white, "size": 14},
            "xanchor": "center",
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 48},
        "len": 0.85,
        "x": 0.15,
        "y": 0.0,
        "steps": slider_steps,
    }],
)

# Style subplot titles.
fig.update_annotations(font=dict(color=off_white, size=16))

# Left axes.
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[dates[0], dates[-1]],
    title_text="Date",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=left_y_range,
    title_text="Process value Xₜ",
)

# Right axes.
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[support_min, support_max],
    title_text=f"Terminal value X_T at T = {YEARS} years",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[0.0, right_y_max],
    title_text="Probability density",
)


###### ______________________________________________________________________________________________________________________________________

##### Time Varying Distribution

Remember, the SDE is just producing a random variable at each time step.

A random variable has a distribution, in this case, it just evolves over time.

Still, the LLN and CLT hold just like for other random variables...

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

YEARS = 15
STEPS_PER_YEAR = 12              # monthly simulation steps
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

N_DISPLAY_PATHS = 30
N_ENSEMBLE_PATHS = 5000

MU = 0.30                        # annual additive drift
SIGMA = 0.22                     # annual additive volatility

FRAME_STRIDE = 3
FRAME_DURATION = 45
INITIAL_I = 3

HIST_BINS = 48
PDF_GRID_SIZE = 600
X_STD_WIDTH = 4.25
DENSITY_HEADROOM = 1.15

OUTPUT_HTML = "/mnt/data/abm_paths_and_increment_distribution.html"
SHOW_FIG = False

# ============================================================
# ABM helpers
# ============================================================

def simulate_abm_ensemble(x0, mu, sigma, n_steps, n_paths, rng):
    z = rng.normal(size=(n_steps, n_paths))
    increments = mu * DT + sigma * np.sqrt(DT) * z

    paths = np.empty((n_steps + 1, n_paths), dtype=float)
    paths[0, :] = x0
    paths[1:, :] = x0 + np.cumsum(increments, axis=0)
    return paths

def abm_level_mean(x0, mu, t):
    return x0 + mu * t

def abm_increment_mean(mu, t):
    return mu * t

def abm_increment_sd(sigma, t):
    return sigma * np.sqrt(t)

def normal_pdf(x, mean, sd):
    x = np.asarray(x, dtype=float)
    if sd <= 0:
        return np.zeros_like(x)
    z = (x - mean) / sd
    return np.exp(-0.5 * z**2) / (sd * np.sqrt(2 * np.pi))

def padded_range(values, pad_fraction=0.08, min_pad=0.10):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [v_min - pad, v_max + pad]

def build_increment_distribution(increments, t, mu, sigma):
    theoretical_mean = abm_increment_mean(mu, t)
    theoretical_sd = abm_increment_sd(sigma, t)

    # x_min and x_max are fixed to [0, 8] as per prompt
    x_min = 0.0
    x_max = 8.0

    bin_edges = np.linspace(x_min, x_max, HIST_BINS + 1)
    hist_density, _ = np.histogram(
        increments,
        bins=bin_edges,
        density=True,
    )

    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_widths = np.diff(bin_edges)

    pdf_x = np.linspace(x_min, x_max, PDF_GRID_SIZE)
    pdf_y = normal_pdf(pdf_x, theoretical_mean, theoretical_sd)

    hist_peak = float(np.max(hist_density)) if hist_density.size else 0.0
    pdf_peak = float(np.max(pdf_y)) if pdf_y.size else 0.0
    y_max = DENSITY_HEADROOM * max(hist_peak, pdf_peak, 1e-9)

    return {
        "bin_centers": bin_centers,
        "bin_widths": bin_widths,
        "hist_density": hist_density,
        "pdf_x": pdf_x,
        "pdf_y": pdf_y,
        "x_range": [x_min, x_max],
        "y_range": [0.0, y_max],
        "mean": theoretical_mean,
    }

# ============================================================
# Time axis and shared ABM ensemble
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)
times = np.arange(N_STEPS + 1) * DT

abm_paths = simulate_abm_ensemble(
    INITIAL_VALUE,
    MU,
    SIGMA,
    N_STEPS,
    N_ENSEMBLE_PATHS,
    rng,
)

display_paths = abm_paths[:, :N_DISPLAY_PATHS]
abm_mean_path = abm_level_mean(INITIAL_VALUE, MU, times)
abm_increments = abm_paths - INITIAL_VALUE

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
mean_color = "#ffaa33"
path_color_over = "#18d618"
path_color_under = "#ff3030"
hist_fill = "rgba(0,212,255,0.48)"
hist_outline = "#00d4ff"
pdf_color = "#ffffff"
zero_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Initial frame data
# ============================================================

initial_end = min(INITIAL_I, N_STEPS)
initial_t = times[initial_end]

initial_distribution = build_increment_distribution(
    abm_increments[initial_end, :],
    initial_t,
    MU,
    SIGMA,
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.08,
    subplot_titles=(
        "Arithmetic Brownian Motion",
        "ABM Increment Distribution",
    ),
)

for j in range(N_DISPLAY_PATHS):
    y_values = display_paths[: initial_end + 1, j]
    current_mean = abm_mean_path[initial_end]
    color = (
        path_color_over
        if y_values[-1] > current_mean
        else path_color_under
    )
    fig.add_trace(
        go.Scatter(
            x=dates[: initial_end + 1],
            y=y_values,
            mode="lines",
            line=dict(color=color, width=1.5),
            opacity=0.50,
            name=(
                f"{N_DISPLAY_PATHS} ABM paths"
                if j == 0
                else f"ABM path {j + 1}"
            ),
            legendgroup="abm-paths",
            showlegend=(j == 0),
            hovertemplate=(
                f"ABM path {j + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Level: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=dates[: initial_end + 1],
        y=abm_mean_path[: initial_end + 1],
        mode="lines",
        line=dict(color=mean_color, width=4),
        name="Theoretical ABM mean",
        legendgroup="abm-mean",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "E[X(t)]: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(color=zero_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=initial_distribution["bin_centers"],
        y=initial_distribution["hist_density"],
        width=initial_distribution["bin_widths"],
        marker=dict(
            color=hist_fill,
            line=dict(color=hist_outline, width=1),
        ),
        name=f"Histogram of {N_ENSEMBLE_PATHS:,} increments",
        legendgroup="increment-histogram",
        showlegend=True,
        hovertemplate=(
            "Increment bin center: %{x:.4f}<br>"
            "Density: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=initial_distribution["pdf_x"],
        y=initial_distribution["pdf_y"],
        mode="lines",
        line=dict(color=pdf_color, width=3),
        name="Theoretical Normal PDF",
        legendgroup="increment-pdf",
        showlegend=True,
        hovertemplate=(
            "Increment: %{x:.4f}<br>"
            "Normal PDF: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[initial_distribution["mean"], initial_distribution["mean"]],
        y=[0.0, initial_distribution["y_range"][1]],
        mode="lines",
        line=dict(color=mean_color, width=3, dash="dash"),
        name="Theoretical mean increment",
        legendgroup="increment-mean",
        showlegend=True,
        hovertemplate="E[X(t) - X₀]: %{x:.4f}<extra></extra>",
    ),
    row=1,
    col=2,
)

fig.add_vline(
    x=0.0,
    line=dict(color=zero_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(INITIAL_I, N_STEPS + 1, FRAME_STRIDE))
if frame_indices[-1] != N_STEPS:
    frame_indices.append(N_STEPS)

n_left_traces = N_DISPLAY_PATHS + 1
n_right_traces = 3
n_total_traces = n_left_traces + n_right_traces

for i in frame_indices:
    frame_name = f"f{i}"
    t = times[i]

    distribution = build_increment_distribution(
        abm_increments[i, :],
        t,
        MU,
        SIGMA,
    )

    frame_data = []

    for j in range(N_DISPLAY_PATHS):
        y_values = display_paths[: i + 1, j]
        current_mean = abm_mean_path[i]
        color = (
            path_color_over
            if y_values[-1] > current_mean
            else path_color_under
        )
        frame_data.append(
            go.Scatter(
                x=dates[: i + 1],
                y=y_values,
                mode="lines",
                opacity=0.50,
                line=dict(color=color, width=1.5),
            )
        )

    frame_data.append(
        go.Scatter(
            x=dates[: i + 1],
            y=abm_mean_path[: i + 1],
            mode="lines",
            line=dict(color=mean_color, width=4),
        )
    )

    frame_data.append(
        go.Bar(
            x=distribution["bin_centers"],
            y=distribution["hist_density"],
            width=distribution["bin_widths"],
        )
    )

    frame_data.append(
        go.Scatter(
            x=distribution["pdf_x"],
            y=distribution["pdf_y"],
            mode="lines",
            line=dict(color=pdf_color, width=3),
        )
    )

    frame_data.append(
        go.Scatter(
            x=[distribution["mean"], distribution["mean"]],
            y=[0.0, distribution["y_range"][1]],
            mode="lines",
            line=dict(color=mean_color, width=3, dash="dash"),
        )
    )

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(n_total_traces)),
            layout=go.Layout(
                xaxis2=dict(range=[0.0, 8.0]), # fixed range for increment x axis
                yaxis2=dict(range=distribution["y_range"]),
            ),
        )
    )

    elapsed_years = i / STEPS_PER_YEAR

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Axis ranges
# ============================================================

left_values = np.r_[display_paths.ravel(), abm_mean_path]
left_y_range = padded_range(left_values, pad_fraction=0.08, min_pad=0.20)

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="Arithmetic Brownian Motion and the Evolution of Its Increment",
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700,
    width=1280,
    margin=dict(t=100, b=120, r=40, l=70),
    barmode="overlay",
    bargap=0.02,
    legend=dict(
        orientation="v",
        x=0.0,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 70},
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": 0.0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {"visible": False},
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 40},
        "len": 0.85,
        "x": 0.15,
        "y": 0.0,
        "steps": slider_steps,
    }],
)

fig.update_annotations(font=dict(color=off_white, size=16))

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[dates[0], dates[-1]],
    title_text="Date",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=left_y_range,
    title_text="ABM level, X(t)",
)

# FIXED: x_range for subplot 2 set to [0, 8]
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[0.0, 8.0],
    title_text="Increment from origin, X(t) - X₀",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=initial_distribution["y_range"],
    title_text="Density",
)


---

##### Geometric Brownian Motion
 
 Step-by-step explicit solution to Geometric Brownian Motion (GBM):
 
 The SDE for GBM is:
 $$
 dS_t = \mu S_t\,dt + \sigma S_t\,dW_t
 $$
 
 1. Take the natural logarithm of both sides: Let $Y_t = \log S_t$.
 
 2. Compute the differential $dY_t$ using Itō's Lemma. Recall that for a function $f(S_t)$,
     $$
     df(S_t) = f'(S_t) dS_t + \frac{1}{2} f''(S_t) (dS_t)^2
     $$
     For $f(S_t) = \log S_t$, we have $f'(S_t) = 1/S_t$, and $f''(S_t) = -1/S_t^2$.
     
     Substitute $dS_t = \mu S_t dt + \sigma S_t dW_t$:
     
     $$
     dY_t = \frac{1}{S_t}\,dS_t - \frac{1}{2}\frac{1}{S_t^2} (dS_t)^2
     $$
     
     Compute $(dS_t)^2$. Because $(dW_t)^2 = dt$, we have:
     $$
     (dS_t)^2 = (\mu S_t dt + \sigma S_t dW_t)^2 = (\sigma S_t)^2 (dW_t)^2 = \sigma^2 S_t^2 dt
     $$
     (ignoring higher-order infinitesimals since $dt^2 \to 0$, $dt\,dW_t \to 0$).
 
     Therefore,
     $$
     dY_t = \frac{1}{S_t} (\mu S_t dt + \sigma S_t dW_t) - \frac{1}{2} \frac{1}{S_t^2} \sigma^2 S_t^2 dt
          = \mu dt + \sigma dW_t - \frac{1}{2}\sigma^2 dt
          = \left(\mu - \frac{1}{2}\sigma^2\right)dt + \sigma dW_t
     $$
 
 3. Integrate both sides from $t=0$ to $t$:
     $$
     \int_0^t dY_s = \int_0^t\left(\mu - \frac{1}{2}\sigma^2\right)ds + \int_0^t \sigma dW_s
     $$
     The left-hand side yields $Y_t - Y_0 = \log S_t - \log S_0$.
     The first integral is $\left(\mu - \frac{1}{2}\sigma^2\right)t$,
     and the second is $\sigma (W_t - W_0)$.
 
 4. Since $W_0 = 0$:
     $$
     \log S_t - \log S_0 = \left(\mu - \frac{1}{2}\sigma^2\right)t + \sigma W_t
     $$
     or equivalently,
     $$
     \log S_t = \log S_0 + \left(\mu - \frac{1}{2}\sigma^2\right)t + \sigma W_t
     $$
 
 5. Exponentiate both sides to solve for $S_t$:
     $$
     S_t = S_0\,\exp\left[\left(\mu - \frac{1}{2}\sigma^2\right)t + \sigma W_t\right]
     $$
 
 6. Distribution: Since $W_t \sim \mathcal{N}(0, t)$, we have
     $$
     \log S_t \sim \mathcal{N}\left(\log S_0 + \left(\mu - \frac{1}{2}\sigma^2\right)t,\ \sigma^2 t\right)
     $$
     So $S_t$ is lognormally distributed:
     $$
     S_t \sim \log\mathcal{N}\left(\log S_0 + \left(\mu - \frac{1}{2}\sigma^2\right)t,\ \sigma^2 t\right)
     $$
 
 7. For direct simulation, let $Z \sim \mathcal{N}(0,1)$ (a standard normal random variable), so $W_t = \sqrt{t}\,Z$:
     $$
     S_t = S_0\,\exp\left[\left(\mu - \frac{1}{2}\sigma^2\right)t + \sigma \sqrt{t}\, Z\right]
     $$

###### ______________________________________________________________________________________________________________________________________

##### Convergence to the Theoretical Distribution for GBM
 
 $S_t \sim \log\mathcal{N}\left(\log S_0 + \left(\mu - \frac{1}{2}\sigma^2\right)t,\, \sigma^2 t\right)$
 
 By the Law of Large Numbers, empirical probabilities, distributions, and statistics of $S_t$ (from many simulated GBM paths) converge to their theoretical (log-normal) values as the number of samples increases.
 
  As $n \to \infty$, for independent simulations $S_t^{(1)},\ldots,S_t^{(n)}$:
  $$
  \overline{\log S_t}^{(n)} = \frac{1}{n} \sum_{i=1}^n \log S_t^{(i)} \xrightarrow{P} \mathbb{E}[\log S_t] = \log S_0 + \left(\mu - \frac{1}{2}\sigma^2\right)t
  $$
  $$
  S^2_{\log S_t} = \frac{1}{n} \sum_{i=1}^n \left( \log S_t^{(i)} - \overline{\log S_t}^{(n)} \right)^2 \xrightarrow{P} \operatorname{Var}[\log S_t] = \sigma^2 t
  $$
  where $\xrightarrow{P}$ denotes convergence in probability.  
  For $S_t$ itself, empirical moments converge to the moments of the lognormal distribution.

In [ ]:
import math
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

YEARS = 5
STEPS_PER_YEAR = 30              # monthly steps
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

# Each animation frame adds one complete path and one terminal sample.
N_PATHS = 200
INITIAL_PATHS = 10

MU = 0.30                        # 30% annual drift
SIGMA = 0.22                     # 22% annual volatility

HIST_BINS = 30
PDF_POINTS = 650
TERMINAL_STD_SPAN = 3.25

FRAME_DURATION = 65
OUTPUT_HTML = "gbm_terminal_distribution_convergence.html"

# For GBM, log axes views are often useful
USE_LOG_Y_LEFT = False
USE_LOG_X_RIGHT = False

# Use SHOW_FIG=0 in the environment to suppress fig.show().
SHOW_FIG = os.getenv("SHOW_FIG", "1") == "1"

# ============================================================
# Simulation and distribution helpers
# ============================================================

def simulate_gbm_paths(s0, mu, sigma, n_steps, n_paths, dt, rng):
    """
    Simulate Geometric Brownian Motion paths:

        dS_t = mu S_t dt + sigma S_t dW_t

    Discretized as (Euler-Maruyama / exact solution):

        S_{t+dt} = S_t * exp[(mu - 0.5*sigma**2)*dt + sigma*sqrt(dt)*Z]

    Returns an array shaped (n_steps + 1, n_paths).
    """
    z = rng.normal(size=(n_steps, n_paths))
    increments = ((mu - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z)
    log_paths = np.vstack([
        np.zeros(n_paths),
        np.cumsum(increments, axis=0),
    ])
    return s0 * np.exp(log_paths)

def gbm_theoretical_mean(s0, mu, times):
    """E[S_t] for GBM."""
    return s0 * np.exp(mu * times)

def gbm_theoretical_std(s0, mu, sigma, times):
    """Stddev of S_t for GBM."""
    # sqrt( (exp(sigma^2 t) - 1) * exp(2 mu t) )
    return s0 * np.exp(mu * times) * np.sqrt(np.exp(sigma ** 2 * times) - 1)

def lognormal_pdf(x, mean, std):
    # mean, std: parameters of the lognormal (in log, not arithmetic)
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    mask = x > 0
    out[mask] = (
        np.exp(-0.5 * ((np.log(x[mask]) - mean) / std) ** 2)
        / (x[mask] * std * np.sqrt(2 * np.pi))
    )
    return out

def lognormal_cdf(x, mean, std):
    x = np.asarray(x, dtype=float)
    z = (np.log(x) - mean) / (std * np.sqrt(2.0))
    return np.where(
        x > 0,
        0.5 * (1.0 + np.vectorize(math.erf)(z)),
        0.0,
    )

def empirical_density(values, bin_edges):
    """Return histogram density heights using fixed, possibly unequal bins."""
    counts, _ = np.histogram(values, bins=bin_edges)
    bin_widths = np.diff(bin_edges)
    return counts / (len(values) * bin_widths)

def padded_range(values, pad_fraction=0.10):
    """Return a padded range, suitable for linear axes."""
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]

    v_min = float(np.min(finite))
    v_max = float(np.max(finite))

    pad = max((v_max - v_min) * pad_fraction, 0.10)
    return [v_min - pad, v_max + pad]

# ============================================================
# Time axis and GBM simulation
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)
times = np.arange(N_STEPS + 1) * DT
terminal_time = YEARS
terminal_date = dates[-1]

gbm_paths = simulate_gbm_paths(
    s0=INITIAL_VALUE,
    mu=MU,
    sigma=SIGMA,
    n_steps=N_STEPS,
    n_paths=N_PATHS,
    dt=DT,
    rng=rng,
)

gbm_mean = gbm_theoretical_mean(
    INITIAL_VALUE,
    MU,
    times,
)
gbm_std = gbm_theoretical_std(INITIAL_VALUE, MU, SIGMA, times)
terminal_values = gbm_paths[-1, :]

# For GBM:
# S_T ~ lognormal(log(s0) + (mu - 0.5*sigma^2)*T, sigma^2 * T)
terminal_log_mean = np.log(INITIAL_VALUE) + (MU - 0.5 * SIGMA ** 2) * terminal_time
terminal_log_std = SIGMA * np.sqrt(terminal_time)
terminal_mean = gbm_theoretical_mean(INITIAL_VALUE, MU, terminal_time)
terminal_std = gbm_theoretical_std(INITIAL_VALUE, MU, SIGMA, terminal_time)

# ============================================================
# Fixed histogram and PDF support
# ============================================================

support_min = min(
    float(np.min(terminal_values)),
    terminal_mean / (np.exp(TERMINAL_STD_SPAN * terminal_log_std)),
)
support_max = max(
    float(np.max(terminal_values)),
    terminal_mean * np.exp(TERMINAL_STD_SPAN * terminal_log_std),
)
hist_edges = np.linspace(support_min, support_max, HIST_BINS + 1)
hist_centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])
hist_widths = np.diff(hist_edges)

pdf_x = np.linspace(support_min, support_max, PDF_POINTS)
pdf_y = lognormal_pdf(
    pdf_x,
    terminal_log_mean,
    terminal_log_std,
)

# Expected theoretical density in each histogram bin:
theoretical_bin_probabilities = np.diff(
    lognormal_cdf(
        hist_edges,
        terminal_log_mean,
        terminal_log_std,
    )
)
theoretical_bin_density = theoretical_bin_probabilities / hist_widths

all_empirical_densities = [
    empirical_density(terminal_values[:n], hist_edges)
    for n in range(INITIAL_PATHS, N_PATHS + 1)
]

maximum_empirical_density = max(
    float(np.max(values)) for values in all_empirical_densities
)

right_y_max = 1.12 * max(
    maximum_empirical_density,
    float(np.max(theoretical_bin_density)),
    float(np.max(pdf_y)),
)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

mean_color = "#ffaa33"   # orange (for GBM, median and mean not same)
pdf_color = "#ffd84d"    # yellow
empirical_hist_color = "#4ea3ff"    # blue
empirical_mean_color = "#00ff88"    # green

path_color_over = "#18d618"         # green
path_color_under = "#ff3030"        # red

baseline_color = "#777777"
newest_terminal_color = "#ffffff"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.55, 0.45],
    horizontal_spacing=0.09,
    subplot_titles=(
        "GBM Sample Paths Added One at a Time",
        f"Terminal Lognormal Distribution at T = {YEARS} Years",
    ),
)

# ------------------------------------------------------------
# Left panel: all complete paths, initially first n visible
# ------------------------------------------------------------

for j in range(N_PATHS):
    terminal_is_over_mean = terminal_values[j] > terminal_mean
    path_color = path_color_over if terminal_is_over_mean else path_color_under

    fig.add_trace(
        go.Scatter(
            x=dates,
            y=gbm_paths[:, j],
            mode="lines",
            line=dict(color=path_color, width=1.35),
            opacity=0.34,
            visible=(j < INITIAL_PATHS),
            name=f"{N_PATHS} GBM sample paths" if j == 0 else f"Path {j + 1}",
            legendgroup="gbm-paths",
            showlegend=(j == 0),
            hovertemplate=(
                f"GBM path {j + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# Theoretical mean path.
fig.add_trace(
    go.Scatter(
        x=dates,
        y=gbm_mean,
        mode="lines",
        line=dict(color=mean_color, width=4),
        name="Theoretical mean path",
        legendgroup="gbm-mean",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "E[S_t]: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

# Bright overlay highlights the newest complete path.
initial_newest_index = INITIAL_PATHS - 1
initial_newest_color = (
    path_color_over
    if terminal_values[initial_newest_index] > terminal_mean
    else path_color_under
)

fig.add_trace(
    go.Scatter(
        x=dates,
        y=gbm_paths[:, initial_newest_index],
        mode="lines",
        line=dict(color=initial_newest_color, width=4),
        opacity=0.95,
        name="Newest path",
        legendgroup="newest-path",
        showlegend=True,
        hovertemplate=(
            f"Newest path: {initial_newest_index + 1}<br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Value: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)
newest_path_trace_index = len(fig.data) - 1

# ------------------------------------------------------------
# Right panel: fixed theoretical law and growing empirical sample
# ------------------------------------------------------------

initial_terminal_sample = terminal_values[:INITIAL_PATHS]
initial_empirical_density = empirical_density(
    initial_terminal_sample,
    hist_edges,
)

# Theoretical histogram bin heights.
fig.add_trace(
    go.Bar(
        x=hist_centers,
        y=theoretical_bin_density,
        width=hist_widths,
        marker=dict(
            color="rgba(255,216,77,0.18)",
            line=dict(color=pdf_color, width=1.2),
        ),
        name="Theoretical bin density",
        legendgroup="theoretical-distribution",
        hovertemplate=(
            "Terminal value: %{x:.4f}<br>"
            "Theoretical bin density: %{y:.6f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# Empirical histogram updated every frame.
fig.add_trace(
    go.Bar(
        x=hist_centers,
        y=initial_empirical_density,
        width=hist_widths * 0.92,
        marker=dict(
            color="rgba(78,163,255,0.68)",
            line=dict(color=empirical_hist_color, width=1.0),
        ),
        name="Empirical terminal density",
        legendgroup="empirical-distribution",
        hovertemplate=(
            "Terminal value: %{x:.4f}<br>"
            "Empirical density: %{y:.6f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)
empirical_hist_trace_index = len(fig.data) - 1

# Smooth theoretical lognormal PDF.
fig.add_trace(
    go.Scatter(
        x=pdf_x,
        y=pdf_y,
        mode="lines",
        line=dict(color=pdf_color, width=4),
        name="Theoretical lognormal PDF",
        legendgroup="theoretical-distribution",
        hovertemplate=(
            "Terminal value: %{x:.4f}<br>"
            "Lognormal PDF: %{y:.6f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# Empirical arithmetic mean marker updated every frame.
initial_empirical_mean = float(np.mean(initial_terminal_sample))
fig.add_trace(
    go.Scatter(
        x=[initial_empirical_mean, initial_empirical_mean],
        y=[0.0, right_y_max],
        mode="lines",
        line=dict(color=empirical_mean_color, width=2.5, dash="dash"),
        name="Empirical terminal mean",
        legendgroup="empirical-mean",
        hovertemplate=(
            "Empirical mean: %{x:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)
empirical_mean_trace_index = len(fig.data) - 1

# Latest terminal observation marker along the baseline.
initial_latest_terminal = float(terminal_values[initial_newest_index])
fig.add_trace(
    go.Scatter(
        x=[initial_latest_terminal],
        y=[0.025 * right_y_max],
        mode="markers",
        marker=dict(
            color=newest_terminal_color,
            size=11,
            symbol="diamond",
            line=dict(color="rgba(0,0,0,0.75)", width=1),
        ),
        name="Newest terminal observation",
        legendgroup="newest-terminal",
        showlegend=False,
        hovertemplate=(
            "Newest terminal observation: %{x:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)
latest_terminal_trace_index = len(fig.data) - 1

# Baseline and terminal markers.
fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1,
)

fig.add_vline(
    x=terminal_date,
    line=dict(color=baseline_color, width=1.2, dash="dash"),
    opacity=0.75,
    row=1,
    col=1,
)

fig.add_vline(
    x=terminal_mean,
    line=dict(color=mean_color, width=2.2, dash="dot"),
    opacity=0.95,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

base_path_trace_indices = list(range(N_PATHS))
animated_trace_indices = base_path_trace_indices + [
    newest_path_trace_index,
    empirical_hist_trace_index,
    empirical_mean_trace_index,
    latest_terminal_trace_index,
]

for n in range(INITIAL_PATHS, N_PATHS + 1):
    newest_index = n - 1
    sample = terminal_values[:n]

    density = empirical_density(sample, hist_edges)
    sample_mean = float(np.mean(sample))
    newest_terminal = float(terminal_values[newest_index])

    newest_color = (
        path_color_over
        if newest_terminal > terminal_mean
        else path_color_under
    )

    frame_data = []

    # Reveal exactly the first n complete paths
    for j in range(N_PATHS):
        frame_data.append(
            go.Scatter(visible=(j < n))
        )

    # Bright newest-path overlay.
    frame_data.append(
        go.Scatter(
            x=dates,
            y=gbm_paths[:, newest_index],
            mode="lines",
            line=dict(color=newest_color, width=4),
            opacity=0.95,
            hovertemplate=(
                f"Newest path: {newest_index + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        )
    )

    # Empirical terminal histogram.
    frame_data.append(
        go.Bar(
            x=hist_centers,
            y=density,
            width=hist_widths * 0.92,
        )
    )

    # Empirical terminal arithmetic mean.
    frame_data.append(
        go.Scatter(
            x=[sample_mean, sample_mean],
            y=[0.0, right_y_max],
            mode="lines",
        )
    )

    # Latest endpoint marker.
    frame_data.append(
        go.Scatter(
            x=[newest_terminal],
            y=[0.025 * right_y_max],
            mode="markers",
        )
    )

    frame_name = f"paths_{n}"

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=animated_trace_indices,
        )
    )

    slider_label = str(n) if (
        n == INITIAL_PATHS
        or n == N_PATHS
        or n % 10 == 0
    ) else ""

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "transition": {"duration": 0},
            },
        ],
        "label": slider_label,
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Axis ranges
# ============================================================

left_values = np.r_[
    gbm_paths.ravel(),
    gbm_mean,
]
left_y_range = padded_range(left_values, pad_fraction=0.07)

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "GBM Monte Carlo Convergence: "
            "Complete Paths to the Terminal Lognormal Distribution"
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=720,
    width=1320,
    margin=dict(t=105, b=135, r=45, l=70),
    barmode="overlay",
    bargap=0.02,
    legend=dict(
        orientation="v",
        x=0.0,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.78)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False,
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                        "mode": "immediate",
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "transition": {"duration": 0},
                        "mode": "immediate",
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 75},
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": 0.0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "visible": True,
            "prefix": "Paths included: ",
            "font": {"color": off_white, "size": 14},
            "xanchor": "center",
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 48},
        "len": 0.85,
        "x": 0.15,
        "y": 0.0,
        "steps": slider_steps,
    }],
)

# Style subplot titles.
fig.update_annotations(font=dict(color=off_white, size=16))

# Left axes.
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[dates[0], dates[-1]],
    title_text="Date",
)

left_axis_kwargs = dict(
    axis_style,
    row=1,
    col=1,
    title_text="Process value Sₜ",
)

if USE_LOG_Y_LEFT:
    left_axis_kwargs.update(
        type="log",
        range=np.log10(left_y_range),
    )
else:
    left_axis_kwargs.update(range=left_y_range)

fig.update_yaxes(**left_axis_kwargs)

# Right axes.
right_x_axis_kwargs = dict(
    axis_style,
    row=1,
    col=2,
    title_text=f"Terminal value S_T at T = {YEARS} years",
)

if USE_LOG_X_RIGHT:
    right_x_axis_kwargs.update(
        type="log",
        range=np.log10([support_min, support_max]),
    )
else:
    right_x_axis_kwargs.update(range=[support_min, support_max])

fig.update_xaxes(**right_x_axis_kwargs)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[0.0, right_y_max],
    title_text="Probability density",
)

if SHOW_FIG:
    fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Time Varying Distribution

Remember, the SDE is just producing a random variable at each time step.

A random variable has a distribution, in this case, it just evolves over time.

Still, the LLN and CLT hold just like for other random variables...

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statistics import NormalDist

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

YEARS = 5
STEPS_PER_YEAR = 30              # monthly simulation steps
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

# One shared GBM ensemble:
# - the first N_DISPLAY_PATHS are plotted on the left
# - all N_ENSEMBLE_PATHS form the increment histogram on the right
N_DISPLAY_PATHS = 30
N_ENSEMBLE_PATHS = 5000

MU = 0.30                        # annual GBM drift
SIGMA = 0.22                     # annual GBM volatility

FRAME_STRIDE = 3                 # reveal quarterly
FRAME_DURATION = 45
INITIAL_I = 3

HIST_BINS = 48
PDF_GRID_SIZE = 700
LOWER_QUANTILE = 0.001
UPPER_QUANTILE = 0.999
DENSITY_HEADROOM = 1.15

OUTPUT_HTML = "/mnt/data/gbm_paths_and_increment_distribution.html"
SHOW_FIG = False

# ============================================================
# GBM helpers
# ============================================================

def simulate_gbm_ensemble(s0, mu, sigma, n_steps, n_paths, rng):
    """
    Simulate one shared Geometric Brownian Motion ensemble:

        dS_t = mu S_t dt + sigma S_t dW_t

    Returns
    -------
    paths : ndarray, shape (n_steps + 1, n_paths)
        Each column is one GBM path.
    """
    z = rng.normal(size=(n_steps, n_paths))

    log_increments = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * z
    )

    paths = np.empty((n_steps + 1, n_paths), dtype=float)
    paths[0, :] = s0
    paths[1:, :] = s0 * np.exp(np.cumsum(log_increments, axis=0))
    return paths


def gbm_arithmetic_mean(s0, mu, t):
    """E[S_t]."""
    return s0 * np.exp(mu * t)


def gbm_geometric_mean(s0, mu, sigma, t):
    """
    exp(E[log S_t]), equal to the median of the GBM level distribution.
    """
    return s0 * np.exp((mu - 0.5 * sigma**2) * t)


def shifted_lognormal_pdf(increment, s0, mu, sigma, t):
    """
    Exact PDF of the GBM increment:

        I_t = S_t - S_0

    Because S_t is lognormal, I_t is shifted lognormal with support
    increment > -S_0.
    """
    increment = np.asarray(increment, dtype=float)
    pdf = np.zeros_like(increment)

    if t <= 0:
        return pdf

    shifted_level = increment + s0
    valid = shifted_level > 0

    log_mean = np.log(s0) + (mu - 0.5 * sigma**2) * t
    log_sd = sigma * np.sqrt(t)

    log_level = np.log(shifted_level[valid])

    pdf[valid] = (
        np.exp(
            -0.5 * ((log_level - log_mean) / log_sd) ** 2
        )
        / (
            shifted_level[valid]
            * log_sd
            * np.sqrt(2 * np.pi)
        )
    )

    return pdf


def shifted_lognormal_quantile(probability, s0, mu, sigma, t):
    """Theoretical quantile of S_t - S_0."""
    z = NormalDist().inv_cdf(probability)

    level_quantile = s0 * np.exp(
        (mu - 0.5 * sigma**2) * t
        + sigma * np.sqrt(t) * z
    )

    return level_quantile - s0


def padded_range(values, pad_fraction=0.08, min_pad=0.10):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))

    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)

    return [max(0.0, v_min - pad), v_max + pad]


def build_increment_distribution(increments, t, s0, mu, sigma):
    """
    Build the empirical GBM-increment histogram and exact
    shifted-lognormal PDF for one horizon t.
    """
    arithmetic_mean_increment = gbm_arithmetic_mean(s0, mu, t) - s0
    geometric_mean_increment = gbm_geometric_mean(
        s0,
        mu,
        sigma,
        t,
    ) - s0

    empirical_low, empirical_high = np.quantile(
        increments,
        [LOWER_QUANTILE, UPPER_QUANTILE],
    )

    theoretical_low = shifted_lognormal_quantile(
        LOWER_QUANTILE,
        s0,
        mu,
        sigma,
        t,
    )
    theoretical_high = shifted_lognormal_quantile(
        UPPER_QUANTILE,
        s0,
        mu,
        sigma,
        t,
    )

    x_min = min(float(empirical_low), theoretical_low)
    x_max = max(float(empirical_high), theoretical_high)

    # Respect the exact support I_t > -S_0 and add modest visual padding.
    span = max(x_max - x_min, 1e-8)
    x_min = max(-0.999 * s0, x_min - 0.04 * span)
    x_max = x_max + 0.06 * span

    bin_edges = np.linspace(x_min, x_max, HIST_BINS + 1)

    hist_density, _ = np.histogram(
        increments,
        bins=bin_edges,
        density=True,
    )

    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_widths = np.diff(bin_edges)

    pdf_x = np.linspace(x_min, x_max, PDF_GRID_SIZE)
    pdf_y = shifted_lognormal_pdf(
        pdf_x,
        s0,
        mu,
        sigma,
        t,
    )

    hist_peak = (
        float(np.max(hist_density))
        if hist_density.size
        else 0.0
    )
    pdf_peak = (
        float(np.max(pdf_y))
        if pdf_y.size
        else 0.0
    )

    y_max = DENSITY_HEADROOM * max(
        hist_peak,
        pdf_peak,
        1e-9,
    )

    return {
        "bin_centers": bin_centers,
        "bin_widths": bin_widths,
        "hist_density": hist_density,
        "pdf_x": pdf_x,
        "pdf_y": pdf_y,
        "x_range": [x_min, x_max],
        "y_range": [0.0, y_max],
        "arithmetic_mean_increment": arithmetic_mean_increment,
        "geometric_mean_increment": geometric_mean_increment,
    }


# ============================================================
# Time axis and shared GBM ensemble
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)
times = np.arange(N_STEPS + 1) * DT

gbm_paths = simulate_gbm_ensemble(
    INITIAL_VALUE,
    MU,
    SIGMA,
    N_STEPS,
    N_ENSEMBLE_PATHS,
    rng,
)

# The displayed paths are members of the same ensemble used
# for the histogram.
display_paths = gbm_paths[:, :N_DISPLAY_PATHS]

gbm_arithmetic_mean_path = gbm_arithmetic_mean(
    INITIAL_VALUE,
    MU,
    times,
)
gbm_geometric_mean_path = gbm_geometric_mean(
    INITIAL_VALUE,
    MU,
    SIGMA,
    times,
)

# Increment from the common origin for every ensemble member.
gbm_increments = gbm_paths - INITIAL_VALUE

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

arithmetic_mean_color = "#00ff88"
geometric_mean_color = "#ffd84d"

path_color_over = "#18d618"
path_color_under = "#ff3030"

hist_fill = "rgba(0,212,255,0.48)"
hist_outline = "#00d4ff"
pdf_color = "#ffffff"
zero_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Initial frame data
# ============================================================

initial_end = min(INITIAL_I, N_STEPS)
initial_t = times[initial_end]

initial_distribution = build_increment_distribution(
    gbm_increments[initial_end, :],
    initial_t,
    INITIAL_VALUE,
    MU,
    SIGMA,
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.08,
    subplot_titles=(
        "Geometric Brownian Motion",
        "GBM Increment Distribution",
    ),
)

# ------------------------------------------------------------
# Left: 30 paths from the shared GBM ensemble
# ------------------------------------------------------------

for j in range(N_DISPLAY_PATHS):
    y_values = display_paths[: initial_end + 1, j]
    current_arithmetic_mean = gbm_arithmetic_mean_path[initial_end]

    color = (
        path_color_over
        if y_values[-1] > current_arithmetic_mean
        else path_color_under
    )

    fig.add_trace(
        go.Scatter(
            x=dates[: initial_end + 1],
            y=y_values,
            mode="lines",
            line=dict(color=color, width=1.5),
            opacity=0.50,
            name=(
                f"{N_DISPLAY_PATHS} GBM paths"
                if j == 0
                else f"GBM path {j + 1}"
            ),
            legendgroup="gbm-paths",
            showlegend=(j == 0),
            hovertemplate=(
                f"GBM path {j + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Level: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=dates[: initial_end + 1],
        y=gbm_arithmetic_mean_path[: initial_end + 1],
        mode="lines",
        line=dict(
            color=arithmetic_mean_color,
            width=4,
        ),
        name="Theoretical arithmetic mean",
        legendgroup="gbm-arithmetic-mean",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "E[S(t)]: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dates[: initial_end + 1],
        y=gbm_geometric_mean_path[: initial_end + 1],
        mode="lines",
        line=dict(
            color=geometric_mean_color,
            width=4,
            dash="dash",
        ),
        name="Theoretical geometric mean",
        legendgroup="gbm-geometric-mean",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "exp(E[log S(t)]): %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(
        color=zero_color,
        width=1,
        dash="dash",
    ),
    opacity=0.65,
    row=1,
    col=1,
)

# ------------------------------------------------------------
# Right: increment histogram from the same GBM ensemble
# ------------------------------------------------------------

fig.add_trace(
    go.Bar(
        x=initial_distribution["bin_centers"],
        y=initial_distribution["hist_density"],
        width=initial_distribution["bin_widths"],
        marker=dict(
            color=hist_fill,
            line=dict(
                color=hist_outline,
                width=1,
            ),
        ),
        name=f"Histogram of {N_ENSEMBLE_PATHS:,} increments",
        legendgroup="increment-histogram",
        showlegend=True,
        hovertemplate=(
            "Increment bin center: %{x:.4f}<br>"
            "Density: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=initial_distribution["pdf_x"],
        y=initial_distribution["pdf_y"],
        mode="lines",
        line=dict(
            color=pdf_color,
            width=3,
        ),
        name="Theoretical shifted-lognormal PDF",
        legendgroup="increment-pdf",
        showlegend=True,
        hovertemplate=(
            "Increment: %{x:.4f}<br>"
            "PDF: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[
            initial_distribution["arithmetic_mean_increment"],
            initial_distribution["arithmetic_mean_increment"],
        ],
        y=[
            0.0,
            initial_distribution["y_range"][1],
        ],
        mode="lines",
        line=dict(
            color=arithmetic_mean_color,
            width=3,
        ),
        name="Arithmetic mean increment",
        legendgroup="increment-arithmetic-mean",
        showlegend=True,
        hovertemplate=(
            "E[S(t) - S₀]: %{x:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[
            initial_distribution["geometric_mean_increment"],
            initial_distribution["geometric_mean_increment"],
        ],
        y=[
            0.0,
            initial_distribution["y_range"][1],
        ],
        mode="lines",
        line=dict(
            color=geometric_mean_color,
            width=3,
            dash="dash",
        ),
        name="Geometric mean increment",
        legendgroup="increment-geometric-mean",
        showlegend=True,
        hovertemplate=(
            "exp(E[log S(t)]) - S₀: %{x:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_vline(
    x=0.0,
    line=dict(
        color=zero_color,
        width=1,
        dash="dash",
    ),
    opacity=0.65,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(
    range(
        INITIAL_I,
        N_STEPS + 1,
        FRAME_STRIDE,
    )
)
if frame_indices[-1] != N_STEPS:
    frame_indices.append(N_STEPS)

n_left_traces = N_DISPLAY_PATHS + 2
n_right_traces = 4
n_total_traces = n_left_traces + n_right_traces

for i in frame_indices:
    frame_name = f"f{i}"
    t = times[i]

    distribution = build_increment_distribution(
        gbm_increments[i, :],
        t,
        INITIAL_VALUE,
        MU,
        SIGMA,
    )

    frame_data = []

    # Left: reveal the same 30 GBM paths through horizon t.
    for j in range(N_DISPLAY_PATHS):
        y_values = display_paths[: i + 1, j]
        current_arithmetic_mean = gbm_arithmetic_mean_path[i]

        color = (
            path_color_over
            if y_values[-1] > current_arithmetic_mean
            else path_color_under
        )

        frame_data.append(
            go.Scatter(
                x=dates[: i + 1],
                y=y_values,
                mode="lines",
                opacity=0.50,
                line=dict(
                    color=color,
                    width=1.5,
                ),
            )
        )

    frame_data.append(
        go.Scatter(
            x=dates[: i + 1],
            y=gbm_arithmetic_mean_path[: i + 1],
            mode="lines",
            line=dict(
                color=arithmetic_mean_color,
                width=4,
            ),
        )
    )

    frame_data.append(
        go.Scatter(
            x=dates[: i + 1],
            y=gbm_geometric_mean_path[: i + 1],
            mode="lines",
            line=dict(
                color=geometric_mean_color,
                width=4,
                dash="dash",
            ),
        )
    )

    # Right: histogram of S_t - S_0 from the exact same ensemble.
    frame_data.append(
        go.Bar(
            x=distribution["bin_centers"],
            y=distribution["hist_density"],
            width=distribution["bin_widths"],
        )
    )

    frame_data.append(
        go.Scatter(
            x=distribution["pdf_x"],
            y=distribution["pdf_y"],
            mode="lines",
            line=dict(
                color=pdf_color,
                width=3,
            ),
        )
    )

    frame_data.append(
        go.Scatter(
            x=[
                distribution["arithmetic_mean_increment"],
                distribution["arithmetic_mean_increment"],
            ],
            y=[
                0.0,
                distribution["y_range"][1],
            ],
            mode="lines",
            line=dict(
                color=arithmetic_mean_color,
                width=3,
            ),
        )
    )

    frame_data.append(
        go.Scatter(
            x=[
                distribution["geometric_mean_increment"],
                distribution["geometric_mean_increment"],
            ],
            y=[
                0.0,
                distribution["y_range"][1],
            ],
            mode="lines",
            line=dict(
                color=geometric_mean_color,
                width=3,
                dash="dash",
            ),
        )
    )

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(n_total_traces)),
            layout=go.Layout(
                xaxis2=dict(
                    range=distribution["x_range"],
                ),
                yaxis2=dict(
                    range=distribution["y_range"],
                ),
            ),
        )
    )

    elapsed_years = i / STEPS_PER_YEAR

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {
                    "duration": 0,
                    "redraw": True,
                },
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Axis ranges
# ============================================================

left_values = np.r_[
    display_paths.ravel(),
    gbm_arithmetic_mean_path,
    gbm_geometric_mean_path,
]

left_y_range = padded_range(
    left_values,
    pad_fraction=0.08,
    min_pad=0.20,
)

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="Geometric Brownian Motion and the Evolution of Its Increment",
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700,
    width=1280,
    margin=dict(
        t=100,
        b=120,
        r=40,
        l=70,
    ),
    barmode="overlay",
    bargap=0.02,
    legend=dict(
        orientation="v",
        x=0.0,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": True,
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": True,
                        },
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {
            "r": 10,
            "t": 70,
        },
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": 0.0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "visible": False,
        },
        "transition": {
            "duration": 0,
        },
        "pad": {
            "b": 10,
            "t": 40,
        },
        "len": 0.85,
        "x": 0.15,
        "y": 0.0,
        "steps": slider_steps,
    }],
)

fig.update_annotations(
    font=dict(
        color=off_white,
        size=16,
    )
)

# Left axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[
        dates[0],
        dates[-1],
    ],
    title_text="Date",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=left_y_range,
    title_text="GBM level, S(t)",
)

# Right axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=initial_distribution["x_range"],
    title_text="Increment from origin, S(t) - S₀",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=initial_distribution["y_range"],
    title_text="Density",
)

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
 - This notebook frames investing around the inescapable trade-off between *risk and reward*: to earn returns you must take risk, there is no free lunch and no silver-bullet strategy, and real wealth is compounded slowly rather than won on a single lucky trade. We measure reward with the expected return $\mathbb{E}[r]$ and risk with the standard deviation $\sigma$, while stressing that both are backward-looking estimates that need not hold going forward.
 - Not all volatility is rewarded, and large swings are actively *dangerous* over the long run because of volatility drag—$(1 + r_{\text{avg}}) = \exp\!\left(\mu - \tfrac{1}{2}\sigma^2\right)$—so chasing big short-run returns can erode long-run compound growth.
 - Markets are hard to beat but not omniscient: the three forms of the Efficient Market Hypothesis (weak, semi-strong, strong) describe increasing efficiency, yet no market can predict the future, and the equilibrium (current market) price is only the market's *best guess*—a compression of all future risk and return—rather than a statement of true worth.
 - Without counterfactuals, a single trade is *poker at best*: we can never cleanly attribute an outcome to our thesis, so the goal is to be the casino and accumulate wealth patiently, not to bet the house.
 - Diversification helps until it doesn't—in a crisis correlations rush toward one and undiversifiable *market beta* dominates—so genuine diversification requires holding structurally *decorrelated* markets and strategies (including alts and non-securities), where portfolio variance $w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\rho\sigma_1\sigma_2$ makes the role of $\rho$ explicit.
 - There is no universally *best* portfolio—like a gym routine, the right allocation depends entirely on your goals—and performance measures such as the Sharpe ratio, maximum drawdown, and CAGR are strictly backward-looking. The key message: the purpose of modeling and backtesting is **not** prediction but *positioning and survival*—we assess our exposures across regimes so that no single state of the world can wipe us out.

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$